# Net-Payout-Based Duration

## 0. Setup, Imports, Paths, Session


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path
from plot_style import COLORS, set_global_plot_style, style_axes

set_global_plot_style()

# Speicherstruktur fuer Intermediate und Final Output
from project_paths import BASE_DIR, DATA_DIR, CACHE_DATA_DIR

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## 2. Master Panel Construction


In [2]:
# Load annual duration input table (full-history saved as euro500_netpayout)
duration_input = load_parquet("euro500_netpayout").copy()

if "firm_id" not in duration_input.columns:
    raise KeyError("euro500_netpayout must contain firm_id")

# Ensure Module D field is present already at Step 1 input stage.
if "CashSTInvst" not in duration_input.columns:
    duration_input["CashSTInvst"] = np.nan

# Derive year from the annual observation date (fallback: effective_date)
if "date" in duration_input.columns:
    duration_input["date"] = pd.to_datetime(duration_input["date"], errors="coerce")
    duration_input["year"] = duration_input["date"].dt.year
elif "effective_date" in duration_input.columns:
    duration_input["effective_date"] = pd.to_datetime(duration_input["effective_date"], errors="coerce")
    duration_input["year"] = duration_input["effective_date"].dt.year
elif "year" not in duration_input.columns:
    raise KeyError("euro500_netpayout needs date, effective_date, or year")

# Quarterly EURO500 panel is only the mapping target for final output.
base_euro500 = load_parquet("euro500").copy()
if "firm_id" not in base_euro500.columns:
    raise KeyError("euro500 must contain firm_id")

euro500_netpayout = duration_input.copy()

print(
    f"Loaded annual duration input (euro500_netpayout): rows={len(duration_input):,}, "
    f"firm_id={duration_input['firm_id'].nunique():,}"
)
print(
    f"Loaded quarterly mapping base (euro500): rows={len(base_euro500):,}, "
    f"firm_id={base_euro500['firm_id'].nunique():,}"
)
print("Step 1 check: CashSTInvst in duration input =", "CashSTInvst" in duration_input.columns)


Loaded annual duration input (euro500_netpayout): rows=196,964, firm_id=1,186
Loaded quarterly mapping base (euro500): rows=56,500, firm_id=1,248
Step 1 check: CashSTInvst in duration input = True


### 2.1 Firm-Year Master Panel (ME, BE, Assets, Sales, NI, GP, Debt, Dividends, Buybacks, CashSTInvst)


In [3]:

def build_masterpanel_firmyear(euro500_netpayout):
    """
    Build one row per firm_id-year for valuation/state construction.
    Keeps firm metadata (name) and accounting/value fields.
    """
    df = euro500_netpayout.copy()

    df["firm_id"] = df["firm_id"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["firm_id", "year"]).copy()
    df["year"] = df["year"].astype(int)

    for dt_col in ["date", "effective_date"]:
        if dt_col in df.columns:
            df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")

    rename_map = {
        "mcap_eur": "ME",
        "Sales": "sales",
        "NetIncome": "net_income",
        "GrossProfit": "gross_profit",
        "Dividends": "dividends",
        "Buybacks": "buybacks",
    }
    existing_renames = {k: v for k, v in rename_map.items() if k in df.columns}
    if existing_renames:
        df = df.rename(columns=existing_renames)

    cols = [
        "firm_id", "name", "year", "date", "effective_date",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks", "CashSTInvst",
        # Mögliche Emissionsspalten (für Netto-Payout, Paper S. 7, Fn. 6)
        "issuances", "equity_issuances", "share_issuances",
    ]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    value_cols = [
        c for c in [
            "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
            "dividends", "buybacks", "CashSTInvst",
            "issuances", "equity_issuances", "share_issuances",
        ] if c in df.columns
    ]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # one row per firm-year from annual input: prefer year-end observation (latest date in year)
    dedupe_date_cols = [c for c in ["date", "effective_date"] if c in df.columns]
    sort_cols = ["firm_id", "year"] + dedupe_date_cols
    df = df.sort_values(sort_cols)
    df = df.drop_duplicates(subset=["firm_id", "year"], keep="last").reset_index(drop=True)

    # Schritt 1: Speichere den RAW-Zustand vor fillna
    df["buybacks_missing"]  = df["buybacks"].isna()  if "buybacks"  in df.columns else pd.Series(True, index=df.index)
    df["dividends_missing"] = df["dividends"].isna() if "dividends" in df.columns else pd.Series(True, index=df.index)

    # Schritt 2: Dividenden immer 0 wenn fehlend
    # (Dividenden werden konsistent gemeldet wenn vorhanden;
    #  fehlen sie, zahlt die Firma wirklich keine)
    df["dividends"] = df["dividends"].fillna(0.0) if "dividends" in df.columns else 0.0

    # Schritt 3: Buybacks NUR dann auf 0 setzen wenn Dividenden vorhanden sind
    # Logik: Wenn Dividendendaten existieren, wurde die Firma aktiv gepflegt.
    #        Dann ist buybacks=NaN wahrscheinlich eine echte Null.
    #        Wenn Dividenden AUCH fehlen, ist die gesamte Zeile datenmäßig dünn
    #        → buybacks bleibt NaN (führt später zu NaN in PO, dann NaN in Duration)
    if "buybacks" in df.columns:
        df["buybacks"] = np.where(
            ~df["dividends_missing"],          # Dividenden vorhanden: Firma aktiv gepflegt
            df["buybacks"].fillna(0.0),        # → Buyback-NaN ist echte Null
            df["buybacks"]                     # → Buyback-NaN bleibt NaN
        )
    else:
        df["buybacks"] = 0.0

    # ============================================================
    # Netto-Payout: PO = Dividenden + Net Buybacks
    # ============================================================
    # Gonçalves (2020), Gl. 2 (nach Boudoukh et al. 2007):
    #   PO_t = Dividenden_t + Aktienrückkäufe_t − Neuemissionen_t
    #        = Dividenden_t + Net Buybacks_t
    #
    # "Net Buybacks" sind in den LSEG-Quelldaten (TR.BuyBackRepurchasedValue)
    # bereits NETTO ausgewiesen: Bruttorepurchases − Neuemissionen.
    # Negative Werte sind möglich und erwartet für Netto-Emittenten (Firmen,
    # die per saldo mehr neue Aktien ausgeben als zurückkaufen).
    #
    # Variante A (mit expliziter Emissionsspalte, falls vorhanden):
    #   PO = dividends + buybacks - issuances  [redundant sicher, identisches Ergebnis]
    # Variante B (ohne explizite Emissionsspalte, Standardfall):
    #   PO = dividends + buybacks  [korrekt, da buybacks bereits netto]
    #
    # In beiden Fällen: KEINE methodische Abweichung von Gonçalves (2020).
    # Die Spezifikation ist vollständig konsistent mit Gonçalves (2020), Gl. 2,
    # der PO als Dividenden + Rückkäufe − Emissionen definiert, in Anlehnung
    # an Boudoukh et al. (2007).
    # ============================================================
    issuances_col = None
    for candidate in ["issuances", "equity_issuances", "share_issuances"]:
        if candidate in df.columns:
            issuances_col = candidate
            break

    if issuances_col is not None:
        df[issuances_col] = df[issuances_col].fillna(0.0)
        df["PO"] = df["dividends"] + df["buybacks"] - df[issuances_col]
        # Net payout kann nicht negativ sein (Paper: firms distribute resources)
        df["PO"] = df["PO"].clip(lower=0)
        print(f"PO = dividends + buybacks - {issuances_col}  (paper-konform, Fn. 6)")
    else:
        # Buybacks bereits netto in Quelldaten → PO = dividends + buybacks ist korrekt.
        # Keine methodische Abweichung von Gonçalves (2020): LSEG weist
        # TR.BuyBackRepurchasedValue bereits netto aus (Bruttorepurchases − Neuemissionen).
        # Negative buybacks-Werte sind möglich und signalisieren Netto-Emittenten.
        df["PO"] = df["dividends"] + df["buybacks"]
        print("PO = dividends + buybacks  (buybacks bereits netto — paper-konform, Gonçalves 2020 Gl. 2)")

    # THESIS: Negative Net-Buyback-Werte bestätigen, dass buybacks netto vorliegen.
    # Netto-Emittenten (Emissionen > Rückkäufe) haben buybacks < 0 — erwartet und korrekt.
    if "buybacks" in df.columns:
        _neg_bb = int((df["buybacks"] < 0).sum())
        _total_bb_obs = max(1, int(df["buybacks"].notna().sum()))
        if _neg_bb > 0:
            print(f"THESIS: {_neg_bb:,} Firmenjahre mit negativen Net Buybacks "
                  f"({_neg_bb / _total_bb_obs:.1%}) — erwartet für Netto-Emittenten. ✓")
        else:
            print(f"THESIS: Keine negativen Buyback-Werte in {_total_bb_obs:,} Beobachtungen. "
                  f"(Alle Firmen sind Netto-Käufer oder BB = 0.)")

    # Robustheitsvariante: Nur Dividenden (kein Buyback-Datenproblem)
    df["PO_divonly"] = df["dividends"].copy()

    df = df.sort_values(["firm_id", "year"]).reset_index(drop=True)
    lag_vars = [
        c for c in ["ME", "BE", "assets", "sales", "net_income", "gross_profit",
                    "debt", "dividends", "buybacks", "CashSTInvst", "PO", "PO_divonly"]
        if c in df.columns
    ]
    for c in lag_vars:
        df[f"{c}_lag1"] = df.groupby("firm_id")[c].shift(1)

    df["dBE"]      = df["BE"] - df["BE_lag1"]
    df["avgBE"]    = 0.5 * (df["BE"] + df["BE_lag1"])
    df["avgAssets"] = 0.5 * (df["assets"] + df["assets_lag1"])

    df.loc[df["ME"] <= 0, "ME"]         = pd.NA
    df.loc[df["BE"] <= 0, "BE"]         = pd.NA
    df.loc[df["assets"] <= 0, "assets"] = pd.NA
    df.loc[df["sales"] <= 0, "sales"]   = pd.NA
    df.loc[df["debt"] < 0, "debt"]      = pd.NA

    return df


In [4]:
master = build_masterpanel_firmyear(euro500_netpayout)

master.head()

PO = dividends + buybacks  (buybacks bereits netto — paper-konform, Gonçalves 2020 Gl. 2)
THESIS: Keine negativen Buyback-Werte in 16,872 Beobachtungen. (Alle Firmen sind Netto-Käufer oder BB = 0.)


KeyError: 'BE'

### Diagnostik: Buyback-Datenverfügbarkeit


In [ ]:
# ============================================================
# DIAGNOSTIK: Buyback-Datenverfügbarkeit
# Nutzt euro500_netpayout (RAW, vor build_masterpanel_firmyear)
# ============================================================

_bb_col  = next((c for c in ["Buybacks", "buybacks"] if c in euro500_netpayout.columns), None)
_div_col = next((c for c in ["Dividends", "dividends"] if c in euro500_netpayout.columns), None)

if "year" not in euro500_netpayout.columns:
    _diag = euro500_netpayout.copy()
    if "date" in _diag.columns:
        _diag["year"] = pd.to_datetime(_diag["date"], errors="coerce").dt.year
    elif "effective_date" in _diag.columns:
        _diag["year"] = pd.to_datetime(_diag["effective_date"], errors="coerce").dt.year
else:
    _diag = euro500_netpayout.copy()

_diag["_has_bb"]  = _diag[_bb_col].notna()  if _bb_col  else False
_diag["_has_div"] = _diag[_div_col].notna() if _div_col else False

_n = len(_diag)
print("=" * 60)
print("BUYBACK-DATENVERFÜGBARKEIT — ÜBERSICHT")
print("=" * 60)
print(f"Firmenjahre gesamt:                {_n:>8,}")
print(f"Buybacks nicht-NaN:                {_diag['_has_bb'].sum():>8,}  ({_diag['_has_bb'].mean():.1%})")
print(f"Dividenden nicht-NaN:              {_diag['_has_div'].sum():>8,}  ({_diag['_has_div'].mean():.1%})")
print()
_both   = (_diag["_has_bb"] & _diag["_has_div"]).sum()
_divonly= (~_diag["_has_bb"] & _diag["_has_div"]).sum()
_bbonly = (_diag["_has_bb"] & ~_diag["_has_div"]).sum()
_none   = (~_diag["_has_bb"] & ~_diag["_has_div"]).sum()
print(f"Beide vorhanden:                   {_both:>8,}  ({_both/_n:.1%})")
print(f"Nur Dividenden:                    {_divonly:>8,}  ({_divonly/_n:.1%})")
print(f"Nur Buybacks:                      {_bbonly:>8,}  ({_bbonly/_n:.1%})")
print(f"Keines vorhanden:                  {_none:>8,}  ({_none/_n:.1%})")
print()

# Firmen, die NIE Buyback-Daten haben
_never_bb = _diag.groupby("firm_id")["_has_bb"].any()
print(f"Firmen mit NIE Buyback-Daten:      {(~_never_bb).sum():>8,} / {len(_never_bb):,}")

# Jahre mit Buyback-Coverage < 50%
_by_year = _diag.groupby("year")[["_has_bb", "_has_div"]].mean()
_low_cov = _by_year[_by_year["_has_bb"] < 0.5]
if len(_low_cov):
    print(f"\nJahre mit Buyback-Coverage < 50%:")
    print(_low_cov.rename(columns={"_has_bb": "buyback_cov", "_has_div": "dividend_cov"}).round(3))
else:
    print("Kein Jahr mit Buyback-Coverage < 50%")

# --- Plot: Coverage über Zeit ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(_by_year.index, _by_year["_has_div"] * 100, label="Dividenden", color="#1f77b4", linewidth=2)
ax.plot(_by_year.index, _by_year["_has_bb"]  * 100, label="Buybacks",   color="#d62728", linewidth=2, linestyle="--")
ax.axhline(50, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("Jahr")
ax.set_ylabel("Coverage (%)")
ax.set_title("Datenverfügbarkeit: Dividenden vs. Buybacks (Firmenjahre)")
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()
print()
del _diag, _by_year


## 3. Data Validity Checks

Before constructing state variables, this section checks whether all denominator and transformation conditions are valid (e.g., positivity for log terms and `1+x>0` for `log1p`).

Potential violations are rare, so only a small share of observations is excluded through the `safe_log` / `safe_log1p` guards.


In [ ]:
checks = pd.DataFrame({
    
    # valuation
    "ME<=0": (master["ME"] <= 0),
    "BE<=0": (master["BE"] <= 0),
    "sales<=0": (master["sales"] <= 0),
    
    # growth
    "BE_lag1<=0": (master["BE_lag1"] <= 0),
    "assets_lag1<=0": (master["assets_lag1"] <= 0),
    "sales_lag1<=0": (master["sales_lag1"] <= 0),

    # profitability denominators
    "avgBE<=0": (master["avgBE"] <= 0),
    "avgAssets<=0": (master["avgAssets"] <= 0),

    # log1p conditions
    "1+PO/ME<=0": (1 + master["PO"]/master["ME"] <= 0),
    "1+(PO+dBE)/BE_lag1<=0": (1 + (master["PO"] + master["dBE"]) / master["BE_lag1"] <= 0),
    "1+NI/avgBE<=0": (1 + master["net_income"] / master["avgBE"] <= 0),
    "1+GP/avgAssets<=0": (1 + master["gross_profit"] / master["avgAssets"] <= 0),

    # leverage denominator
    "debt+ME<=0": (master["debt"] + master["ME"] <= 0),
})

checks.mean().sort_values(ascending=False)

## 4. Construction of Firm-Level State Variables


In this step, we construct the firm-level state variables used in the pooled VAR(1).

The state vector contains exactly 12 variables:
- `bm`, `py`, `sy`, `beg`, `ag`, `sg`, `csprof`, `roe`, `gp`, `lev`, `blev`, `cash`

The clean-surplus profitability definition is:
$$
csprof_{i,t} = \log\left(1 + \frac{PO_{i,t} + \Delta BE_{i,t}}{BE_{i,t-1}}\right).
$$

At this stage, variables are constructed from raw accounting/market inputs; winsorization is applied in Step 4.


In [ ]:
def build_state_variables(master):
    df = master.copy()

    def safe_log(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index, dtype=float)
        m = x > 0
        out.loc[m] = np.log(x.loc[m])
        return out

    def safe_log1p(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index, dtype=float)
        m = x > -1  # log(1+x) defined iff 1+x > 0
        out.loc[m] = np.log1p(x.loc[m])
        return out

    # Ensure numeric inputs
    num_cols = [
        "BE", "ME", "PO", "sales",
        "BE_lag1", "assets", "assets_lag1", "sales_lag1",
        "dBE", "net_income", "avgBE", "gross_profit", "avgAssets",
        "debt", "CashSTInvst"
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Valuation
    df["bm"] = safe_log(df["BE"] / df["ME"])
    df["py"] = safe_log1p(df["PO"] / df["ME"])
    df["sy"] = safe_log(df["sales"] / df["ME"])

    # Growth
    df["beg"] = safe_log(df["BE"] / df["BE_lag1"])
    df["ag"]  = safe_log(df["assets"] / df["assets_lag1"])
    df["sg"]  = safe_log(df["sales"] / df["sales_lag1"])

    # Profitability
    csprof_raw = (df["PO"] + df["dBE"]) / df["BE_lag1"]
    roe_raw    = df["net_income"] / df["avgBE"]
    gp_raw     = df["gross_profit"] / df["avgAssets"]

    # Optional paper-near floor so log1p stays well-defined
    csprof_raw = csprof_raw.clip(lower=-0.99)
    roe_raw    = roe_raw.clip(lower=-0.99)
    gp_raw     = gp_raw.clip(lower=-0.99)

    df["csprof"] = safe_log1p(csprof_raw)
    df["roe"]    = safe_log1p(roe_raw)
    df["gp"]     = safe_log1p(gp_raw)

    # Robustheitsvarianten: Payout Yield und csprof nur aus Dividenden
    # (buyback-unabhängig; wird nur für Robustheitslauf verwendet, nicht im Haupt-VAR)
    if "PO_divonly" in df.columns:
        df["PO_divonly"] = pd.to_numeric(df["PO_divonly"], errors="coerce")
        df["py_divonly"] = safe_log1p(df["PO_divonly"] / df["ME"])
        csprof_divonly_raw = (df["PO_divonly"] + df["dBE"]) / df["BE_lag1"]
        csprof_divonly_raw = csprof_divonly_raw.clip(lower=-0.99)
        df["csprof_divonly"] = safe_log1p(csprof_divonly_raw)

    # Capital structure: book leverage and cash holdings
    assets = df["assets"].to_numpy(dtype=float)
    debt   = df["debt"].to_numpy(dtype=float)
    cash   = df["CashSTInvst"].to_numpy(dtype=float)
    me     = df["ME"].to_numpy(dtype=float)

    blev = np.full(len(df), np.nan, dtype=float)
    cash_ratio = np.full(len(df), np.nan, dtype=float)

    mask_assets = assets > 0
    blev[mask_assets] = debt[mask_assets] / assets[mask_assets]
    cash_ratio[mask_assets] = cash[mask_assets] / assets[mask_assets]

    df["blev"] = blev
    df["cash"] = cash_ratio

    # Capital structure: market leverage
    denom = debt + me
    lev = np.full(len(df), np.nan, dtype=float)
    mask_lev = denom > 0
    lev[mask_lev] = debt[mask_lev] / denom[mask_lev]
    df["lev"] = lev

    # Final state variable set
    state_vars = [
        "bm", "py", "sy",
        "beg", "ag", "sg",
        "csprof", "roe", "gp",
        "lev", "blev", "cash"
    ]

    df[state_vars] = df[state_vars].replace([np.inf, -np.inf], np.nan)

    return df

In [ ]:
state_panel = build_state_variables(master)

In [ ]:
state_vars = [
    "bm","py","sy",
    "beg","ag","sg",
    "csprof","roe","gp",
    "lev","blev","cash"
]

state_panel[state_vars].isna().mean().sort_values(ascending=False)
state_panel[state_vars].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).T

In [ ]:
state_panel.groupby("year")[state_vars].count().tail()

## 5. Winsorizing State Variables

State variables are winsorized by year at the 1st and 99th percentiles for robustness before VAR estimation.


In [ ]:

# ============================================================
# Winsorize State Variables by YEAR (1% / 99%)
# Paper S. 11, Fn. 7: "I winsorize each non-bounded variable in
# the state vector at 1% and 99% percentiles for each cross-section."
# ============================================================

# lev, blev, cash werden bewusst NICHT winsorisiert (paper-konform):
# Diese Variablen sind durch ihre Definition in [0,1] beschränkt
# (market leverage, book leverage, cash ratio) und brauchen daher
# keine zusätzliche Winsorisierung. Paper Fn. 7 spricht explizit
# von "non-bounded variables".
STATE_VARS_WINSOR = [
    "bm", "py", "sy",
    "beg", "ag", "sg",
    "csprof", "roe", "gp"
]

def winsorize_by_year(df, cols, year_col="year", lower=0.01, upper=0.99):
    """
    Jahresweise Winsorisierung (cross-section, kein look-ahead bias).
    Paper S. 11, Fn. 7: "this avoids any look-ahead bias in the winsorization"
    """
    df = df.copy()

    for col in cols:
        lower_bound = df.groupby(year_col)[col].transform(
            lambda x: x.quantile(lower)
        )
        upper_bound = df.groupby(year_col)[col].transform(
            lambda x: x.quantile(upper)
        )
        df[col] = df[col].clip(lower_bound, upper_bound)

    return df


state_panel = winsorize_by_year(
    state_panel,
    STATE_VARS_WINSOR,
    year_col="year"
)

print(f"Winsorisiert: {STATE_VARS_WINSOR}")
print("lev, blev, cash: nicht winsorisiert (bounded in [0,1], paper-konform)")


## 6. Estimation of the Firm-Level State VAR

We estimate a pooled VAR(1) on the 12-dimensional state vector using equation-by-equation OLS, without state standardization.

Model form:
$$
x_{t+1} = c + \Phi x_t + u_{t+1}.
$$

The implementation includes a stability check via eigenvalues of `Phi` and forecasting utilities based on the same raw-state transition equation.


In [ ]:

# ============================================================
# STEP 5: Firm-Level State VAR(1) — Fama-MacBeth, expanding window
# Gonçalves (2020), S. 13 & Fn. 9; Internet Appendix IA.A
#
# Problem 3 (fix): expanding window — parameters for year t estimated
#                  only on data with year < t (no look-ahead bias)
# Problem 4 (fix): Fama-MacBeth — cross-sectional OLS per year t',
#                  slopes weighted by N_{t'} (Paper S. 13, Fn. 9)
# Problem 5 (fix): intercept calibrated to cross-sectional medians:
#                  const = (I − Φ) @ xbar_target,
#                  xbar_target = time-average of annual cross-sectional medians
# Problem 8 (fix): min_T = 2 (paper S. 9: "minimum of two previous years")
# ============================================================

import statsmodels.api as sm   # kept for any external use; core estimation uses numpy

# ----------------------------
# 5.1 Full state vector (12-dimensional)
# ----------------------------
var_states = [
    "bm", "py", "sy",
    "beg", "ag", "sg",
    "csprof", "roe", "gp",
    "lev", "blev", "cash",
]

TIME_COL = "year"
k = len(var_states)
lead_cols = [f"{v}_lead" for v in var_states]

# ----------------------------
# 5.2 Build VAR sample
#     Each row: (firm, t) with state at t and state at t+1
# ----------------------------
df_var = state_panel.sort_values(["firm_id", TIME_COL]).copy()

for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("firm_id")[v].shift(-1)

req_cols = var_states + lead_cols
df_var = df_var.dropna(subset=req_cols).copy()

# Problem 8: minimum 2 consecutive observations per firm (paper S. 9)
min_T = 2
firm_counts = df_var.groupby("firm_id").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["firm_id"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms : {df_var['firm_id'].nunique()}")
print(f"  obs   : {len(df_var)}")
print(f"  years : {df_var[TIME_COL].min()} – {df_var[TIME_COL].max()}")

# ----------------------------
# 5.3 Helper functions for Fama-MacBeth VAR estimation
# ----------------------------

def _fama_macbeth_phi(train_df, var_states, time_col, min_cs_obs=None):
    """
    Fama-MacBeth slope matrix.

    For each year t' in train_df, run cross-sectional OLS:
        X_{j,t'+1} = c_{t'} + Φ_{t'} X_{j,t'} + u_{j,t'+1}

    Returns Φ as the N-weighted average of Φ_{t'} across years,
    and the list of per-year sample sizes.
    """
    k = len(var_states)
    lc = [f"{v}_lead" for v in var_states]
    if min_cs_obs is None:
        min_cs_obs = k + 1          # bare minimum for OLS identification

    Phi_list, N_list = [], []

    for _, grp in train_df.groupby(time_col):
        if len(grp) < min_cs_obs:
            continue
        X_t  = grp[var_states].to_numpy(dtype=float)   # (N, k)
        X_t1 = grp[lc].to_numpy(dtype=float)           # (N, k)

        # OLS with intercept:  [1 | X_t] @ β = X_t1
        # β shape: (k+1, k); β[0,:] = intercepts, β[1:,:] = Φ^T
        X_d = np.column_stack([np.ones(len(grp)), X_t])
        try:
            beta, _, _, _ = np.linalg.lstsq(X_d, X_t1, rcond=None)
        except np.linalg.LinAlgError:
            continue

        Phi_list.append(beta[1:, :].T)   # (k, k)
        N_list.append(len(grp))

    if not Phi_list:
        return None, []

    total_N = sum(N_list)
    Phi = sum(n * P for n, P in zip(N_list, Phi_list)) / total_N
    return Phi, N_list


def _calibrate_intercept(Phi, train_df, var_states, time_col):
    """
    Calibrate intercept so that the model's implied steady state equals
    the time-average of annual cross-sectional medians (Paper S. 13):

        const = (I − Φ) @ xbar_target
        xbar_target = mean over years of median(X_{:,t'})
    """
    k = len(var_states)
    annual_medians = (
        train_df.groupby(time_col)[var_states]
        .median()
        .to_numpy(dtype=float)
    )                                   # (T_train, k)
    xbar_target = annual_medians.mean(axis=0)   # (k,)
    const = (np.eye(k) - Phi) @ xbar_target
    return const, xbar_target


def _compute_sigma(train_df, var_states, Phi, const):
    """
    Covariance matrix Σ from firm-demeaned VAR(1) residuals (Paper S. 13):

        u_{j,t} = X_{j,t+1} − const − Φ X_{j,t}
        Σ = Cov(u_tilde),  u_tilde = firm-demeaned u
    """
    k = len(var_states)
    lc = [f"{v}_lead" for v in var_states]

    X_t   = train_df[var_states].to_numpy(dtype=float)     # (N, k)
    X_t1  = train_df[lc].to_numpy(dtype=float)             # (N, k)
    firms = train_df["firm_id"].to_numpy()

    # Raw residuals
    resid = X_t1 - const[np.newaxis, :] - (Phi @ X_t.T).T  # (N, k)

    # Firm-demean using numpy groupby via unique
    unique_f, f_idx = np.unique(firms, return_inverse=True)
    firm_sums  = np.zeros((len(unique_f), k))
    firm_cnts  = np.bincount(f_idx)
    np.add.at(firm_sums, f_idx, resid)
    firm_means = firm_sums / firm_cnts[:, np.newaxis]       # (n_firms, k)

    resid_dm = resid - firm_means[f_idx]                    # (N, k)

    Sigma = (resid_dm.T @ resid_dm) / len(resid_dm)
    return Sigma


# ----------------------------
# 5.4 Expanding-window Fama-MacBeth estimation
# ----------------------------

def estimate_var_expanding(df_var, var_states, time_col="year", min_cs_obs=None):
    """
    Fama-MacBeth VAR(1) with expanding window.

    For each year t, parameters are estimated using all observations
    with year < t (no look-ahead bias).

    Returns
    -------
    var_params_by_year : dict {year: (Phi, const, Sigma)}
        Phi  : (k, k)  transition matrix
        const: (k,)    intercept calibrated to cross-sectional medians
        Sigma: (k, k)  residual covariance from firm-demeaned residuals
    """
    all_years = sorted(df_var[time_col].unique())
    params = {}

    for t in all_years:
        train = df_var[df_var[time_col] < t]
        if train.empty:
            continue

        Phi, N_list = _fama_macbeth_phi(train, var_states, time_col, min_cs_obs)
        if Phi is None:
            continue

        const, xbar_target = _calibrate_intercept(Phi, train, var_states, time_col)
        Sigma = _compute_sigma(train, var_states, Phi, const)

        params[t] = (Phi, const, Sigma)

    return params


print("\nEstimating Fama-MacBeth expanding-window VAR(1)…")
var_params_by_year = estimate_var_expanding(df_var, var_states, time_col=TIME_COL)
print(f"  Parameters available for {len(var_params_by_year)} years: "
      f"{min(var_params_by_year)} – {max(var_params_by_year)}")

# ----------------------------
# 5.5 Diagnostics: latest-year parameters
# ----------------------------
latest_year = max(var_params_by_year.keys())
Phi, const, Sigma = var_params_by_year[latest_year]

I_k = np.eye(k)

try:
    xbar = np.linalg.solve(I_k - Phi, const)
    xbar_s = pd.Series(xbar, index=var_states, name="steady_state")
except np.linalg.LinAlgError:
    xbar, xbar_s = None, None

eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print(f"\nStability diagnostics (params from training data up to year {latest_year}):")
print(f"  Max |eigenvalue| of Φ : {max_eig:.4f}")
if max_eig >= 1.0:
    print("  WARNING: VAR not stable — duration code applies θ-shrinkage (IA.A).")
else:
    print("  VAR stable.")

phi_df   = pd.DataFrame(Phi,  index=var_states, columns=var_states)
const_s  = pd.Series(const,   index=var_states, name="const")

print("\nIntercepts (calibrated to cross-sectional medians):")
display(const_s.round(4))

print(f"\nTransition matrix Φ (latest year = {latest_year}):")
display(phi_df.round(4))

if xbar_s is not None:
    print("\nImplied steady state (= xbar_target by construction):")
    display(xbar_s.round(4))

# ----------------------------
# 5.6 Forecast utilities
#     Accept optional Phi_ / const_ overrides so Cell 21 can pass
#     year-specific parameters without modifying global state.
# ----------------------------

def row_to_state_vector(row, states=var_states):
    """Convert one row of state_panel / df_var into a state vector x_t."""
    return row[states].to_numpy(dtype=float)


def forecast_states_iterative(x0, H, Phi_=None, const_=None):
    """
    Iterative forecast:  x_{t+h} = const + Φ x_{t+h-1}
    Returns array of shape (H, k), horizons h = 1 … H.
    """
    _Phi   = Phi   if Phi_   is None else Phi_
    _const = const if const_ is None else const_
    out, x = np.zeros((H, len(x0))), x0.copy()
    for h in range(H):
        x = _const + _Phi @ x
        out[h] = x
    return out


def forecast_states_closedform(x0, H, Phi_=None, const_=None):
    """
    Closed-form forecast:  x_{t+h} = Φ^h x0 + (I − Φ^h) xbar
    Requires a stable VAR with invertible (I − Φ).
    Returns array of shape (H, k), horizons h = 1 … H.
    """
    _Phi   = Phi   if Phi_   is None else Phi_
    _const = const if const_ is None else const_
    _xbar  = np.linalg.solve(I_k - _Phi, _const)
    out = np.zeros((H, len(x0)))
    for h in range(1, H + 1):
        Ph = np.linalg.matrix_power(_Phi, h)
        out[h - 1] = Ph @ x0 + (I_k - Ph) @ _xbar
    return out


def forecast_dataframe_from_row(row, H, method="iterative", Phi_=None, const_=None):
    """Convenience wrapper — returns DataFrame with forecasted states for h = 1…H."""
    x0 = row_to_state_vector(row)
    if method == "iterative":
        arr = forecast_states_iterative(x0, H, Phi_, const_)
    elif method == "closedform":
        arr = forecast_states_closedform(x0, H, Phi_, const_)
    else:
        raise ValueError("method must be 'iterative' or 'closedform'")
    out = pd.DataFrame(arr, columns=var_states)
    out.insert(0, "horizon", np.arange(1, H + 1))
    return out


## 7. Duration Construction

This step computes `Duration_NP` in the spirit of Goncalves (2020): for each firm-year, expected net payouts are generated from the Step-5 VAR and the firm-specific discount rate is solved from the valuation identity `ME/BE = sum_h E[PO_{t+h}/BE_t] * exp(-h*dr)`.

Duration is then computed as the Macaulay-style weighted average maturity of expected net payouts under the solved firm-year discount rate.


In [ ]:

# ============================================================
# STEP 6: Equity Duration with Jensen-inequality corrections
# Gonçalves (2020), Eq. 6–7, IA.3–IA.6
#
# Problem 1 (fix): Add Jensen corrections v1(h) and v2(h):
#   E_t[exp(py_{t+h})]        = exp(E_t[py_{t+h}]   + v1(h))
#   E_t[exp(cumBEg_{t+1..t+h})] = exp(E_t[cumBEg_h]  + h·v2(h))
#
# Problem 6 (fix): Increase horizon with geometric tail (IA.7):
#   H_explicit = 30 (matrix-power regime)
#   Tail from H+1 to ∞ via geometric series at steady-state BEg+v2
#
# Problem 3/4/5 integration: use year-specific (Phi, const, Sigma)
#   from var_params_by_year[t] for each firm-year t.
# ============================================================

# ----------------------------
# 6.0 Parameters & global steady-state values
# ----------------------------
H = 30           # explicit horizon; tail handles h > H
THETA = 0.1 ** (1.0 / 10.0)   # θ^10 = 0.1  (IA section c), θ ≈ 0.7943

# Selector indices in var_states (12-dimensional)
IDX_PY     = var_states.index("py")      # log payout-to-book ratio (kept for reference)
IDX_BEG    = var_states.index("beg")     # log book-equity growth
IDX_CSPROF = var_states.index("csprof")  # log profitability (cash-based)

# e_po = 1_csprof − 1_beg  (Paper IA.1: "1_po" selector)
# This reflects PO/BE = exp(csprof) / exp(beg) − 1 ≈ csprof − beg in logs
e_po  = np.zeros(k); e_po[IDX_CSPROF] = 1.0; e_po[IDX_BEG] = -1.0
e_beg = np.zeros(k); e_beg[IDX_BEG]   = 1.0   # 1_BEg in paper notation

# Global steady-state values from latest-year VAR (for diagnostics and fallbacks)
if xbar_s is not None:
    beg_ss    = float(xbar_s["beg"])
    csprof_ss = float(xbar_s["csprof"]) if "csprof" in xbar_s.index else np.nan
    py_ss     = float(xbar_s["py"])     if "py"     in xbar_s.index else np.nan
else:
    beg_ss = csprof_ss = py_ss = np.nan
g_ss = beg_ss   # backward-compatibility alias

# Numerical guards
TAIL_SPREAD_MIN = 1e-4   # minimum (dr − beg_ss − v2_bar) for stable geometric tail
DURATION_MAX    = 1_000.0


# ----------------------------
# 6.1 Jensen corrections v1(h) and v2(h)
#     via IA.3–IA.6 recursions
# ----------------------------

def compute_v_corrections(Phi, Sigma, H, theta=THETA):
    """
    Compute Jensen-inequality corrections v1(h), v2(h) for h = 1 … H.

    v1(h) [IA.3]: corrects E[exp((1_csprof−1_beg)'·x_{t+h})] = exp(E[...] + v1(h))
        where 1_po = 1_csprof − 1_beg  (Paper IA.1)
        F1(1) = I,  F1(h) = F1(h−1)·Γ + I  (Γ = theta-shrunk Phi)
        v1(h) = v1(h−1) + 0.5·e_po'Γ^{h−1}ΣΓ'^{h−1}e_po
                         + e_po'Γ^{h−1}Σ F1(h)'e_beg
        boundary: v1(0) = 0

    v2(h) [IA.4]: corrects E[exp(cumBEg_{1..h})] = exp(E[cumBEg_h] + h·v2(h))
        h·v2(h) = (h−1)·v2(h−1) + 0.5·CovBEg(h,h) + Σ_{τ=1}^{h−1} CovBEg(τ,h)
        CovBEg(τ,h) = e_beg'·F2(τ,h)·e_beg  [IA.5–6]
        F2(τ,h) = Γ·F2(τ−1,h) + Σ·Γ'^{h−τ},  F2(0,h) = 0

    Γ = theta·Phi  (slopes shrunk toward 0; IA section c)
    Note: theta-shrinkage applies ONLY to v-corrections, NOT to state forecasts.
    """
    G = theta * Phi     # Γ: theta-shrunk slope matrix (Phi_ss = 0)

    # Precompute powers G^0 … G^H
    G_pow = [np.eye(k)]
    for _ in range(H):
        G_pow.append(G @ G_pow[-1])

    # ---- v1 ----
    v1 = np.zeros(H + 1)   # v1[0]=0 (boundary); v1[h] for h=1..H
    F1 = np.eye(k)          # F1(1) = I

    for h in range(1, H + 1):
        Gh1 = G_pow[h - 1]                       # G^{h−1}
        # 0.5 · e_po' G^{h−1} Σ G'^{h−1} e_po
        term_a = 0.5 * (e_po @ Gh1 @ Sigma @ Gh1.T @ e_po)
        # e_po' G^{h−1} Σ F1(h)' e_beg
        term_b = e_po @ Gh1 @ Sigma @ F1.T @ e_beg
        v1[h] = v1[h - 1] + term_a + term_b
        F1 = F1 @ G + np.eye(k)                  # F1(h+1) = F1(h)·G + I

    # ---- v2 ----
    v2 = np.zeros(H + 1)   # v2[0]=0; v2[h] for h=1..H
    hv2_prev = 0.0

    for h in range(1, H + 1):
        # Build CovBEg(τ, h) for τ = 1 … h via F2 recursion
        F2 = np.zeros((k, k))
        cov_beg = []
        for tau in range(1, h + 1):
            # F2(τ,h) = G·F2(τ−1,h) + Σ·G'^{h−τ}
            exp_idx = h - tau
            F2 = G @ F2 + Sigma @ G_pow[exp_idx].T
            cov_beg.append(float(e_beg @ F2 @ e_beg))

        # h·v2(h) = (h−1)·v2(h−1) + 0.5·CovBEg(h,h) + Σ_{τ=1}^{h−1} CovBEg(τ,h)
        cross = sum(cov_beg[: h - 1])      # Σ_{τ=1}^{h−1} CovBEg(τ,h)
        hv2_new = hv2_prev + 0.5 * cov_beg[-1] + cross
        v2[h] = hv2_new / h
        hv2_prev = hv2_new

    return v1[1:], v2[1:]   # arrays of length H, index [h−1] → horizon h


# ----------------------------
# 6.2 Year-specific parameter lookup
# ----------------------------

def _get_var_params(year):
    """
    Return (Phi, const, Sigma) for the given year from var_params_by_year.
    Falls back to the most recent available year ≤ target, or earliest overall.
    """
    if year in var_params_by_year:
        return var_params_by_year[year]
    available = sorted(var_params_by_year.keys())
    past = [y for y in available if y <= year]
    fallback = past[-1] if past else available[0]
    return var_params_by_year[fallback]


def _theta_phi(Phi, theta=THETA):
    """Apply theta-shrinkage to Phi (IA section c: Phi_ss = 0)."""
    return theta * Phi


# ----------------------------
# 6.3 Corrected payout and BE forecasts
# ----------------------------

def forecast_book_equity_and_payouts(row, H=H, year=None, method="iterative"):
    """
    Build h-step forecasts of (csprof−beg) and cumulative BEg using year-specific
    VAR parameters, then compute expected PO_{t+h}/BE_t with
    Jensen corrections v1(h) and v2(h).

    Jensen-corrected payout (Paper Eq. 6 / IA.1):
      E_t[PO_{t+h}/BE_t] = [exp((1_csprof−1_beg)' x_{t+h|t} + v1(h)) − 1]
                           · exp(Σ_{τ=1}^h e_beg' x_{t+τ|t} + h·v2(h))

    Notes:
      - The selector is e_po = e_csprof − e_beg  (not e_py).
      - The −1 arises from PO/BE = exp(csprof)/exp(beg) − 1 in the paper's log-linear model.
      - cumBEg is the sum up to h (not h−1); no lag.
      - hv2 = h · v2(h) is the full Jensen correction for cumulative BEg up to h.
      - State forecasts use the FULL Phi_y (not theta-shrunk); theta-shrinkage
        applies only within compute_v_corrections (Paper IA, section c).
    """
    if year is None:
        year = int(row.get(TIME_COL, latest_year))

    Phi_y, const_y, Sigma_y = _get_var_params(year)

    v1_arr, v2_arr = compute_v_corrections(Phi_y, Sigma_y, H)

    x0      = row_to_state_vector(row)
    # Bug fix: use full Phi_y for state forecasts; theta-shrinkage is only for v-corrections
    fstates = forecast_states_iterative(x0, H, Phi_=Phi_y, const_=const_y)

    # Corrected payout selector: e_po = e_csprof − e_beg  (Paper IA.1)
    csprof_minus_beg = fstates @ e_po    # shape (H,): (csprof−beg)_{t+h|t}
    beg_hat          = fstates[:, IDX_BEG]
    py_hat           = fstates[:, IDX_PY]   # kept for diagnostics only
    cumBEg           = np.cumsum(beg_hat)   # Σ_{τ=1}^h beg_{t+τ|t}  (up to h, no lag)
    hv2              = np.arange(1, H + 1) * v2_arr   # h · v2(h)

    # E_t[PO_{t+h}/BE_t] = [exp(csprof_minus_beg + v1) − 1] · exp(cumBEg + h·v2)
    po_over_be0 = (np.exp(csprof_minus_beg + v1_arr) - 1.0) * np.exp(cumBEg + hv2)

    be0 = pd.to_numeric(pd.Series([row.get("BE", np.nan)]), errors="coerce").iloc[0]
    if np.isfinite(be0) and be0 > 0:
        be0    = float(be0)
        be_hat = be0 * np.exp(cumBEg)
        be_lag = np.concatenate([[be0], be_hat[:-1]])
    else:
        be_hat = be_lag = np.full(H, np.nan)

    return pd.DataFrame({
        "horizon":     np.arange(1, H + 1),
        "py_hat":      py_hat,
        "beg_hat":     beg_hat,
        "cumBEg":      cumBEg,
        "v1":          v1_arr,
        "v2":          v2_arr,
        "PO_over_BE0": po_over_be0,
        "BE_lag":      be_lag,
        "BE_hat":      be_hat,
    })


# ----------------------------
# 6.4 Geometric tail (IA.7)
# ----------------------------

def add_terminal_value(path_df, dr, Phi_y, Sigma_y, v1_arr, v2_arr, year=None):
    """
    Geometric-tail PV from horizon H+1 to ∞ (IA.7).

    Assumes convergence to steady-state at h = H+1:
      csprof_minus_beg_ss = (e_csprof − e_beg)' xbar  (from year-specific or global VAR)
      beg_ss from xbar of year-specific (or global) VAR.
      ratio = exp(beg_ss + v2_bar − dr)   [must be < 1]

    Tail is valid only when dr > beg_ss + v2_bar (convergence condition).

    Returns (PV_tail, TV_H_over_BE0)
      PV_tail = tail PV as of t=0 in BE_t units
      TV_H    = PV_tail re-expressed at horizon H (for duration numerator)
    """
    _, const_y, _ = _get_var_params(year) if year is not None else (None, const, None)
    try:
        xbar_y              = np.linalg.solve(I_k - Phi_y, const_y)
        csprof_minus_beg_ss = float(e_po  @ xbar_y)   # steady-state (csprof − beg)
        beg_ss_y            = float(e_beg @ xbar_y)
    except np.linalg.LinAlgError:
        csprof_minus_beg_ss = csprof_ss - beg_ss       # global fallback
        beg_ss_y            = beg_ss

    v2_bar  = float(v2_arr[-1])
    ratio   = np.exp(beg_ss_y + v2_bar - dr)
    if ratio >= 1.0 or (dr - beg_ss_y - v2_bar) <= TAIL_SPREAD_MIN:
        return np.nan, np.nan

    cumBEg_H = float(path_df["cumBEg"].iloc[-1])
    v2_H     = float(path_df["v2"].iloc[-1])
    v1_ss    = float(v1_arr[-1])
    H_exp    = int(path_df["horizon"].max())

    # First tail cash-flow (at h = H+1), relative to BE_t
    # [exp(csprof_minus_beg_ss + v1_ss) − 1] · exp(cumBEg_H + H_exp·v2_H) · exp(−dr·(H+1))
    first_tail = ((np.exp(csprof_minus_beg_ss + v1_ss) - 1.0)
                  * np.exp(cumBEg_H + H_exp * v2_H)
                  * np.exp(-dr * (H_exp + 1)))

    PV_tail = first_tail / (1.0 - ratio)
    TV_H    = PV_tail * np.exp(dr * H_exp)
    return PV_tail, TV_H


def present_value_weights(path_df, dr, PV_tail=None):
    """Discount explicit payouts and optionally add geometric-tail PV."""
    out    = path_df.copy()
    h_arr  = out["horizon"].to_numpy(dtype=float)
    out["discount_factor"] = np.exp(dr * h_arr)
    out["PV_PO"]           = out["PO_over_BE0"] / out["discount_factor"]

    pv_explicit = float(np.nansum(out["PV_PO"].to_numpy()))

    if PV_tail is not None and np.isfinite(PV_tail):
        total_pv = pv_explicit + PV_tail
    else:
        total_pv = pv_explicit
        PV_tail  = np.nan

    return out, total_pv, PV_tail


# ----------------------------
# 6.5 Valuation residual and bisection solver
# ----------------------------

def valuation_residual_from_path(path_df, me_be, dr,
                                  Phi_y, Sigma_y, v1_arr, v2_arr, year=None):
    """f(dr) = model-implied ME/BE − observed ME/BE."""
    PV_tail, _ = add_terminal_value(
        path_df, dr, Phi_y, Sigma_y, v1_arr, v2_arr, year=year)
    if not np.isfinite(PV_tail):
        return np.nan
    _, total_pv, _ = present_value_weights(path_df, dr, PV_tail)
    return (total_pv - me_be) if np.isfinite(total_pv) else np.nan


def solve_discount_rate(row, H=H, low=0.005, high=0.60, max_iter=80):
    """Solve dr_{j,t} from ME/BE valuation identity via bisection."""
    me = pd.to_numeric(pd.Series([row.get("ME", np.nan)]), errors="coerce").iloc[0]
    be = pd.to_numeric(pd.Series([row.get("BE", np.nan)]), errors="coerce").iloc[0]
    if not (np.isfinite(me) and np.isfinite(be) and me > 0 and be > 0):
        return np.nan, None

    year    = int(row.get(TIME_COL, latest_year))
    me_be   = float(me) / float(be)
    path_df = forecast_book_equity_and_payouts(row, H=H, year=year)
    v1_arr  = path_df["v1"].to_numpy()
    v2_arr  = path_df["v2"].to_numpy()
    Phi_y, const_y, Sigma_y = _get_var_params(year)

    try:
        xbar_y   = np.linalg.solve(I_k - Phi_y, const_y)
        beg_ss_y = float(e_beg @ xbar_y)
    except np.linalg.LinAlgError:
        beg_ss_y = beg_ss

    # Bug fix: lower bound must also clear v2_bar so the geometric tail converges.
    # Tail is valid only when dr > beg_ss + v2_bar; without this the solver
    # searches a region where add_terminal_value always returns NaN.
    v2_bar  = float(v2_arr[-1])
    eff_low = max(float(low), beg_ss_y + v2_bar + TAIL_SPREAD_MIN)

    def f(dr_val):
        return valuation_residual_from_path(
            path_df, me_be, dr_val, Phi_y, Sigma_y, v1_arr, v2_arr, year=year)

    f_low, f_high = f(eff_low), f(high)

    for _ in range(8):
        if np.isfinite(f_low) and np.isfinite(f_high) and f_low * f_high < 0:
            break
        high   = min(2.0, high * 1.5)
        f_high = f(high)

    # Grid-search fallback
    if not (np.isfinite(f_low) and np.isfinite(f_high) and f_low * f_high < 0):
        grid = np.linspace(eff_low, high, 121)
        vals = np.array([f(x) for x in grid], dtype=float)
        ok   = np.isfinite(vals)
        if not ok.any():
            return np.nan, path_df
        return float(grid[ok][np.argmin(np.abs(vals[ok]))]), path_df

    # Bisection
    a, b, fa = eff_low, high, f_low
    for _ in range(max_iter):
        m  = 0.5 * (a + b)
        fm = f(m)
        if not np.isfinite(fm):
            grid = np.linspace(a, b, 61)
            vals = np.array([f(x) for x in grid], dtype=float)
            ok   = np.isfinite(vals)
            if not ok.any():
                return np.nan, path_df
            return float(grid[ok][np.argmin(np.abs(vals[ok]))]), path_df
        if abs(fm) < 1e-9 or (b - a) < 1e-7:
            return float(m), path_df
        if fa * fm <= 0:
            b = m
        else:
            a, fa = m, fm

    return float(0.5 * (a + b)), path_df


# ----------------------------
# 6.6 Duration (Eq. 7)
# ----------------------------

def duration_from_solved_dr(row, H=H):
    """
    Paper-consistent Duration_NP for one firm-year (Eq. 7):
      Dur = Σ_h h · PV[PO_{t+h}] / total_PV
    Tail contribution treated as lump-sum at H+1.
    """
    dr, path_df = solve_discount_rate(row, H=H)
    if not np.isfinite(dr) or path_df is None:
        return pd.Series({"Duration_NP": np.nan, "discount_rate_NP": np.nan,
                          "pv_check": np.nan, "TV_share": np.nan})

    me = pd.to_numeric(pd.Series([row.get("ME", np.nan)]), errors="coerce").iloc[0]
    be = pd.to_numeric(pd.Series([row.get("BE", np.nan)]), errors="coerce").iloc[0]
    if not (np.isfinite(me) and np.isfinite(be) and me > 0 and be > 0):
        return pd.Series({"Duration_NP": np.nan, "discount_rate_NP": np.nan,
                          "pv_check": np.nan, "TV_share": np.nan})
    me_be = float(me) / float(be)

    year   = int(row.get(TIME_COL, latest_year))
    v1_arr = path_df["v1"].to_numpy()
    v2_arr = path_df["v2"].to_numpy()
    Phi_y, const_y, Sigma_y = _get_var_params(year)

    PV_tail, _ = add_terminal_value(
        path_df, dr, Phi_y, Sigma_y, v1_arr, v2_arr, year=year)
    pv_df, total_pv, PV_tail = present_value_weights(path_df, dr, PV_tail)

    if not (np.isfinite(total_pv) and total_pv > 0):
        return pd.Series({"Duration_NP": np.nan, "discount_rate_NP": np.nan,
                          "pv_check": np.nan, "TV_share": np.nan})

    h_arr        = pv_df["horizon"].to_numpy(dtype=float)
    pv_arr       = pv_df["PV_PO"].to_numpy(dtype=float)
    num_explicit = float(np.nansum(h_arr * pv_arr))
    H_exp        = int(h_arr[-1])
    num_tail     = (H_exp + 1) * PV_tail if np.isfinite(PV_tail) else 0.0

    duration = (num_explicit + num_tail) / total_pv
    tv_share = float(PV_tail / total_pv) if np.isfinite(PV_tail) else np.nan

    if not np.isfinite(duration) or duration <= 0:
        return pd.Series({"Duration_NP": np.nan, "discount_rate_NP": np.nan,
                          "pv_check": np.nan, "TV_share": np.nan})

    return pd.Series({
        "Duration_NP":      duration,
        "discount_rate_NP": dr,
        "pv_check":         total_pv / me_be,
        "TV_share":         tv_share,
    })


# ----------------------------
# 6.7 Apply to panel
# ----------------------------
print("Computing paper-consistent Duration_NP (Jensen-corrected, expanding-window VAR)…")
res = state_panel.apply(duration_from_solved_dr, axis=1, H=H)
state_panel[["Duration_NP", "discount_rate_NP", "pv_check", "TV_share"]] = res

print("\nDuration_NP:")
print(state_panel["Duration_NP"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print(f"NaN share: {state_panel['Duration_NP'].isna().mean():.2%}")

print("\ndiscount_rate_NP:")
print(state_panel["discount_rate_NP"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\npv_check (model ME/BE ÷ observed ME/BE; should be ≈ 1):")
print(state_panel["pv_check"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nTV_share (terminal-value fraction of total PV):")
print(state_panel["TV_share"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

corr_vars = ["Duration_NP", "bm", "py", "ag", "beg"]
avail = [v for v in corr_vars if v in state_panel.columns]
print("\nCorrelation matrix:")
display(state_panel[avail].corr().round(3))


### Robustheitslauf: Duration_NP_divonly (nur Dividenden)


In [ ]:

# ============================================================
# ROBUSTHEIT: Duration_NP_divonly (nur Dividenden als PO)
# Vollständiger VAR+Duration-Lauf mit py_divonly/csprof_divonly
# ============================================================

if "py_divonly" in state_panel.columns and "csprof_divonly" in state_panel.columns:

    # Erstelle eine Kopie des state_panel mit substituierten Variablen
    _sp_div = state_panel.copy()
    _sp_div["py"]     = _sp_div["py_divonly"]
    _sp_div["csprof"] = _sp_div["csprof_divonly"]

    # Winsorisiere die substituierten Variablen analog zu den Haupt-Variablen
    for _v in ["py", "csprof"]:
        _lo = _sp_div.groupby("year")[_v].transform(lambda x: x.quantile(0.01))
        _hi = _sp_div.groupby("year")[_v].transform(lambda x: x.quantile(0.99))
        _sp_div[_v] = _sp_div[_v].clip(lower=_lo, upper=_hi)

    # --------------------------------------------------------
    # DIAG 1: py und csprof nach Substitution + Winsorisierung
    # --------------------------------------------------------
    print("=" * 60)
    print("DIAG 1: py / csprof nach Substitution und Winsorisierung")
    print("=" * 60)
    print(f"  _sp_div shape: {_sp_div.shape}")
    for _col in ["py", "csprof", "py_divonly", "csprof_divonly"]:
        if _col in _sp_div.columns:
            _s = _sp_div[_col]
            print(f"  {_col:<20s}  nicht-NaN: {_s.notna().sum():>6,} / {len(_s):,} "
                  f"({_s.notna().mean():.1%})  "
                  f"Bereich: [{_s.min():.4f}, {_s.max():.4f}]")

    # --------------------------------------------------------
    # FIX 1: Inf-Bereinigung nach Winsorisierung
    # Ursache: Wenn P99 von csprof_divonly selbst inf ist (z.B. weil viele
    # Beobachtungen in einem Jahr inf haben), schlägt clip() nicht an und
    # inf-Werte bleiben erhalten. Sie müssen vor VAR und Solver entfernt werden,
    # da inf im Zustandsvektor die Matrixmultiplikation und Bisektionssuche
    # korrumpiert.
    # --------------------------------------------------------
    _n_inf_py_before = np.isinf(_sp_div["py"].fillna(0)).sum()
    _n_inf_cs_before = np.isinf(_sp_div["csprof"].fillna(0)).sum()

    _sp_div["py"]     = _sp_div["py"].replace([np.inf, -np.inf], np.nan)
    _sp_div["csprof"] = _sp_div["csprof"].replace([np.inf, -np.inf], np.nan)

    _n_inf_py_after = np.isinf(_sp_div["py"].fillna(0)).sum()
    _n_inf_cs_after = np.isinf(_sp_div["csprof"].fillna(0)).sum()

    print(f"\n  Inf-Bereinigung:")
    print(f"  py:     {_n_inf_py_before} inf-Werte entfernt → {_n_inf_py_after} verbleibend "
          f"{'✓' if _n_inf_py_after == 0 else '⚠ noch inf!'}")
    print(f"  csprof: {_n_inf_cs_before} inf-Werte entfernt → {_n_inf_cs_after} verbleibend "
          f"{'✓' if _n_inf_cs_after == 0 else '⚠ noch inf!'}")

    # Lead-Spalten analog zu df_var-Vorbereitung in Step 5
    _sp_div = _sp_div.sort_values(["firm_id", "year"])
    for _v in state_vars:
        _sp_div[f"{_v}_lead"] = _sp_div.groupby("firm_id")[_v].shift(-1)
    _lead_cols = [f"{_v}_lead" for _v in state_vars]
    _sp_div = _sp_div.dropna(subset=state_vars + _lead_cols).copy()

    print(f"\n  _sp_div nach dropna(state_vars + leads): {_sp_div.shape}")
    print(f"  py nicht-NaN nach dropna: {_sp_div['py'].notna().sum():,}")
    print(f"  ME nicht-NaN nach dropna: {_sp_div['ME'].notna().sum():,}")
    print(f"  BE nicht-NaN nach dropna: {_sp_div['BE'].notna().sum():,}")

    print("Schätze VAR (Dividenden-only Robustheitscheck)...")
    _var_div = estimate_var_expanding(_sp_div, state_vars)
    print(f"  VAR-Schätzjahre: {len(_var_div)}")

    # Duration-Lauf (identisch zur Hauptrechnung, nur andere VAR-Parameter)
    _saved_var = var_params_by_year.copy()
    var_params_by_year.clear()
    var_params_by_year.update(_var_div)

    # --------------------------------------------------------
    # DIAG 3: VAR-Parameter-Quelle prüfen
    # --------------------------------------------------------
    print("\n" + "=" * 60)
    print("DIAG 3: var_params_by_year nach Update mit _var_div")
    print("=" * 60)
    print(f"  Anzahl Jahre in var_params_by_year (div-only): {len(var_params_by_year)}")
    print(f"  Anzahl Jahre in _saved_var (Haupt-VAR):        {len(_saved_var)}")
    if var_params_by_year:
        _yr_last = sorted(var_params_by_year.keys())[-1]
        _Pd, _cd, _Sd = var_params_by_year[_yr_last]
        print(f"  div-only VAR, letztes Jahr {_yr_last}: "
              f"Phi shape={_Pd.shape}, ||const||={np.linalg.norm(_cd):.4f}")
    _common_years = sorted(set(var_params_by_year.keys()) & set(_saved_var.keys()))
    if _common_years:
        _yr_cmp = _common_years[-1]
        _P_main, _, _ = _saved_var[_yr_cmp]
        _P_div2, _, _ = var_params_by_year[_yr_cmp]
        _params_differ = not np.allclose(_P_main, _P_div2)
        print(f"  Phi(main) ≠ Phi(div-only) für Jahr {_yr_cmp}: "
              f"{'JA ✓ (div-only Params korrekt geladen)' if _params_differ else 'NEIN — identisch! div-only Params fehlen!'}")

    # --------------------------------------------------------
    # DIAG 2: Solver-Testlauf auf 20-Zeilen-Slice
    # --------------------------------------------------------
    print("\n" + "=" * 60)
    print("DIAG 2: Solver-Test auf erstem 20-Zeilen-Slice")
    print("=" * 60)
    _test_mask  = _sp_div["py"].notna() & _sp_div["csprof"].notna() & \
                  _sp_div["ME"].notna() & _sp_div["BE"].notna()
    _test_slice = _sp_div[_test_mask].head(20).copy()
    print(f"  Test-Slice: {len(_test_slice)} Zeilen")

    _test_res = _test_slice.apply(duration_from_solved_dr, axis=1)

    print(f"  Rückgabetyp von apply():  {type(_test_res).__name__}")
    if hasattr(_test_res, "columns"):
        print(f"  Spalten in Ergebnis:      {list(_test_res.columns)}")
        _n_valid = _test_res["Duration_NP"].notna().sum()
        print(f"  Duration_NP nicht-NaN:    {_n_valid} / {len(_test_res)}")
        print(_test_res[["Duration_NP", "discount_rate_NP", "pv_check"]].head(5).round(3).to_string())
    else:
        print(f"  WARNUNG: Ergebnis ist kein DataFrame. Inhalt: {_test_res}")

    # --------------------------------------------------------
    # DIAG 4: Payout-Forecast für Beispiel-Firma (nur wenn Solver NaN liefert)
    # --------------------------------------------------------
    _solver_all_nan = (not hasattr(_test_res, "columns")) or _test_res["Duration_NP"].isna().all()
    if _solver_all_nan:
        print("\n" + "=" * 60)
        print("DIAG 4: Solver liefert NaN — Payout-Forecast-Diagnose")
        print("=" * 60)
        _ex_row = _test_slice.iloc[0]
        _ex_fid = _ex_row.get("firm_id", "?")
        _ex_yr  = int(_ex_row.get("year", 0))
        _ex_me  = _ex_row.get("ME", float("nan"))
        _ex_be  = _ex_row.get("BE", float("nan"))
        print(f"  Beispiel-Firma: firm_id={_ex_fid}, year={_ex_yr}, "
              f"ME={_ex_me:.2f}, BE={_ex_be:.2f}, ME/BE={_ex_me/_ex_be:.3f}")
        print(f"  py={_ex_row.get('py', float('nan')):.4f}, "
              f"csprof={_ex_row.get('csprof', float('nan')):.4f}, "
              f"beg={_ex_row.get('beg', float('nan')):.4f}")
        try:
            _ex_path = forecast_book_equity_and_payouts(_ex_row, H=5, year=_ex_yr)
            print(f"\n  E[PO/BE_0] für h=1..5 (mit div-only VAR):")
            print(_ex_path[["horizon", "py_hat", "beg_hat", "PO_over_BE0"]].round(5).to_string(index=False))
            _po_vals = _ex_path["PO_over_BE0"].values
            if np.all(np.isnan(_po_vals)):
                print("  → ALLE PO_over_BE0 sind NaN: Problem liegt im Forecast, nicht im Solver.")
            elif np.all(_po_vals <= 0):
                print("  → ALLE PO_over_BE0 ≤ 0: negative erwartete Ausschüttungen (Wachstumsfirma?).")
            else:
                print(f"  → PO_over_BE0 Min/Max: {_po_vals.min():.5f} / {_po_vals.max():.5f}")
        except Exception as _e_fcast:
            print(f"  Fehler in forecast_book_equity_and_payouts: {_e_fcast}")
    else:
        print("\n  → Solver liefert valide Werte auf Test-Slice. NaN-Problem liegt NACH dem apply().")

    # --------------------------------------------------------
    # DIAG EXTRA: Lambda-Bug-Check (NACH apply, VOR Zuweisung)
    # --------------------------------------------------------
    print("\n" + "=" * 60)
    print("DIAG EXTRA: Typ-Analyse _results_div und Lambda-Verhalten")
    print("=" * 60)
    _sp_div["Duration_NP_divonly"] = None
    _results_div = _sp_div.apply(duration_from_solved_dr, axis=1)

    print(f"  type(_results_div): {type(_results_div).__name__}")
    if hasattr(_results_div, "columns"):
        print(f"  Spalten: {list(_results_div.columns)}")
        print(f"  Duration_NP nicht-NaN: {_results_div['Duration_NP'].notna().sum():,} / {len(_results_div):,}")
        _col0 = _results_div.iloc[:, 0]
        print(f"\n  isinstance(_results_div.iloc[:,0], dict) = {isinstance(_col0, dict)}")
        print(f"  → apply(lambda x: ... isinstance(x,dict) ...) iteriert über SPALTEN, nicht Zeilen!")
        _lambda_out = _results_div.apply(
            lambda x: x.get("Duration_NP") if isinstance(x, dict) else np.nan
        )
        print(f"\n  Lambda-Ergebnis (sollte Duration_NP-Werte sein, ist stattdessen):")
        print(f"  {_lambda_out.to_dict()}")
        print(f"\n  URSACHE DES NaN-PROBLEMS: lambda prüft isinstance(x, dict),")
        print(f"  aber apply(axis=1) gibt pd.Series zurück (nicht dict).")
        print(f"  Das apply ohne axis-Angabe iteriert dann über SPALTEN.")
        print(f"  FIX: _results_div['Duration_NP'] direkt verwenden.")
    else:
        print(f"  Kein DataFrame — unerwarteter Rückgabetyp: {type(_results_div)}")

    # Ergebnisse korrekt extrahieren (nutze direkte Spaltenzuweisung)
    if hasattr(_results_div, "columns") and "Duration_NP" in _results_div.columns:
        _sp_div["Duration_NP_divonly"] = _results_div["Duration_NP"]
    else:
        _sp_div["Duration_NP_divonly"] = _results_div.apply(
            lambda x: x.get("Duration_NP") if isinstance(x, dict) else np.nan
        )

    # Wiederherstellen der originalen VAR-Parameter
    var_params_by_year.clear()
    var_params_by_year.update(_saved_var)

    # Ergebnisse in state_panel übertragen.
    # Vorhandene Spalte zuerst entfernen, um _x/_y-Suffix-Konflikt bei
    # wiederholter Zellenausführung zu vermeiden.
    state_panel.drop(columns=["Duration_NP_divonly"], errors="ignore", inplace=True)
    state_panel = state_panel.merge(
        _sp_div[["firm_id", "year", "Duration_NP_divonly"]].drop_duplicates(["firm_id", "year"]),
        on=["firm_id", "year"],
        how="left"
    )

    print(f"\n  state_panel Zeilen nach Merge: {len(state_panel):,}")
    print(f"  Duration_NP_divonly nicht-NaN: {state_panel['Duration_NP_divonly'].notna().sum():,}")

    # --------------------------------------------------------
    # DIAG 5: Direkter Richtungsvergleich Duration_NP vs. Duration_NP_divonly
    # Erwartung: divonly-Duration sollte bei buyback-zahlenden Firmen LÄNGER sein,
    # weil Dividenden < Netto-Payout → geringere Ausschüttungsrendite → höhere Duration.
    # Wenn median(divonly) < median(main), deutet das auf einen Richtungsfehler hin.
    # --------------------------------------------------------
    _both_diag = state_panel[["Duration_NP", "Duration_NP_divonly"]].dropna()
    if len(_both_diag) > 0:
        print("\n" + "=" * 60)
        print("DIAG 5: Richtungsvergleich Duration_NP vs. Duration_NP_divonly")
        print("=" * 60)
        _desc = pd.concat(
            [_both_diag["Duration_NP"].describe(percentiles=[.05, .25, .5, .75, .95]),
             _both_diag["Duration_NP_divonly"].describe(percentiles=[.05, .25, .5, .75, .95])],
            axis=1
        ).round(2)
        print(_desc.to_string())

        _share_longer = (_both_diag["Duration_NP_divonly"] > _both_diag["Duration_NP"]).mean()
        _med_main = _both_diag["Duration_NP"].median()
        _med_div  = _both_diag["Duration_NP_divonly"].median()
        _mean_diff = (_both_diag["Duration_NP_divonly"] - _both_diag["Duration_NP"]).mean()

        print(f"\n  Anteil Obs mit Duration_NP_divonly > Duration_NP: {_share_longer:.1%}")
        print(f"  Erwartet: >50% (Dividenden < Net Payout → divonly länger)")
        print(f"  Mittlere Differenz (divonly − main): {_mean_diff:+.2f} Jahre")
        if _med_div >= _med_main:
            print(f"  Median divonly ({_med_div:.1f}) ≥ Median main ({_med_main:.1f}) — "
                  f"Richtung korrekt ✓")
        else:
            print(f"  WARNUNG: Median divonly ({_med_div:.1f}) < Median main ({_med_main:.1f})")
            print(f"  → Möglicher Richtungsfehler in der Substitutionslogik.")
            print(f"  → Prüfen: py_divonly und csprof_divonly korrekt berechnet?")

    # --- Vergleich: Pearson & Spearman ---
    from scipy import stats as _scipy_stats

    _both_valid = state_panel[["Duration_NP", "Duration_NP_divonly"]].dropna()
    n_both = len(_both_valid)
    print(f"\nTHESIS: Beobachtungen mit beiden Duration-Werten: {n_both:,}")

    if n_both > 1:
        _pearson_r  = _both_valid.corr().iloc[0, 1]
        _spearman_r = _scipy_stats.spearmanr(
            _both_valid["Duration_NP"], _both_valid["Duration_NP_divonly"]
        ).statistic
        print(f"THESIS: Pearson-Korrelation  (Duration_NP, Duration_NP_divonly) = {_pearson_r:.3f}")
        print(f"THESIS: Spearman-Korrelation (Duration_NP, Duration_NP_divonly) = {_spearman_r:.3f}")
        if _spearman_r >= 0.85:
            print(f"→ Spearman ≥ 0.85: Duration-Ranking robust gegenüber Buyback-Datenlücke ✓")
        else:
            print(f"→ WARNUNG: Spearman < 0.85 ({_spearman_r:.3f}) — Buybacks beeinflussen "
                  f"Duration-Rankings erheblich. Buyback-Datenlücken prüfen.")

        display(_both_valid.corr().round(3))

        # Streudiagramm (Zufallsstichprobe bis 1.000 Beobachtungen)
        _sample = _both_valid.sample(min(1000, n_both), random_state=42)
        _xy_max = max(_sample["Duration_NP"].max(), _sample["Duration_NP_divonly"].max()) * 1.05
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(
            _sample["Duration_NP"], _sample["Duration_NP_divonly"],
            s=8, alpha=0.35, color=COLORS.get("blue_light", "#6baed6")
        )
        ax.plot([0, _xy_max], [0, _xy_max],
                color=COLORS.get("red", "#d62728"),
                linewidth=1.2, linestyle="--", label="45°-Linie")
        ax.set_xlabel("Duration_NP (mit Buybacks)")
        ax.set_ylabel("Duration_NP (nur Dividenden)")
        ax.set_title(f"Robustheit: Buyback vs. Dividend-only Duration\n"
                     f"(n = {min(1000, n_both):,} Zufallsstichprobe, "
                     f"Spearman ρ = {_spearman_r:.3f})")
        ax.legend()
        style_axes(ax)
        plt.tight_layout()
        plt.show()
    else:
        print("WARNUNG: Zu wenige gemeinsame Beobachtungen für Korrelationsberechnung.")
        print(f"  Duration_NP nicht-NaN: {state_panel['Duration_NP'].notna().sum():,}")
        print(f"  Duration_NP_divonly nicht-NaN: {state_panel['Duration_NP_divonly'].notna().sum():,}")

    del _sp_div, _var_div, _results_div, _saved_var, _both_valid

else:
    print("SKIP: py_divonly / csprof_divonly nicht in state_panel verfügbar.")
    print("Bitte erst build_state_variables() mit dem aktualisierten Code ausführen.")
    state_panel["Duration_NP_divonly"] = np.nan


## 8. Build Final Output Table


In [ ]:

# ============================================================
# STEP 7: Build firm-year output table
# Output mirrors EQDuration_Implied.parquet structure:
# one row per firm_id × year, merged in regression notebook
# by firm_id + year_pred (= event_year - 1).
# ============================================================

out_cols = ["firm_id", "year", "date"] + [
    c for c in ["Duration_NP", "discount_rate_NP", "pv_check", "TV_share",
                "Duration_NP_divonly",
                "bm", "py", "sy", "beg", "ag", "sg", "csprof",
                "roe", "gp", "lev", "blev", "cash", "ME", "BE"]
    if c in state_panel.columns
]

out = (
    state_panel[out_cols]
    .copy()
    .rename(columns={"Duration_NP": "Duration_NetPayout"})
    .sort_values(["firm_id", "year"])
    .drop_duplicates(subset=["firm_id", "year"], keep="last")
    .reset_index(drop=True)
)

# ----------------------------
# 7.1 dr_constrained flag
# Flag observations where the solver hit the year-specific lower bound
# (beg_ss_y + v2_bar + TAIL_SPREAD_MIN). These are growth firms with
# ME/BE so high that the model cannot match it even at the minimum
# feasible discount rate. They should be excluded from regressions.
# ----------------------------
_year_eff_low = {}
for _yr, (_Phi, _const, _Sig) in var_params_by_year.items():
    try:
        _xbar = np.linalg.solve(I_k - _Phi, _const)
        _bss  = float(e_beg @ _xbar)
    except np.linalg.LinAlgError:
        _bss  = beg_ss
    _, _v2 = compute_v_corrections(_Phi, _Sig, H)
    _year_eff_low[_yr] = max(0.005, _bss + float(_v2[-1]) + TAIL_SPREAD_MIN)

out["_dr_low"] = out["year"].map(_year_eff_low)
# Constrained if dr landed within 1bp of the lower bound
out["dr_constrained"] = out["discount_rate_NP"] <= (out["_dr_low"] + 1e-4)
out.drop(columns=["_dr_low"], inplace=True)

n_constrained = out["dr_constrained"].sum()
print(f"dr_constrained: {n_constrained:,} / {len(out):,} obs ({n_constrained/len(out):.1%})")

# ----------------------------
# 7.1a pv_fit_poor: schlechte Modell-Anpassung
# pv_check misst model_ME_BE / observed_ME_BE; sollte ≈ 1 sein.
# Werte weit unter 1: Modell kann das hohe ME/BE nicht erklären.
# ----------------------------
PV_FIT_THRESHOLD = 0.85
out["pv_fit_poor"] = (
    out["pv_check"].notna() & (out["pv_check"] < PV_FIT_THRESHOLD)
)
n_poor = int(out["pv_fit_poor"].sum())
print(f"pv_fit_poor (<{PV_FIT_THRESHOLD}): {n_poor:,} / {len(out):,} obs "
      f"({n_poor/len(out):.1%})")

# ----------------------------
# 7.1b Buyback/Dividenden-Flags aus Master übernehmen
# ----------------------------
flag_cols = [c for c in ["buybacks_missing", "dividends_missing"] if c in master.columns]
if flag_cols:
    out = out.merge(
        master[["firm_id", "year"] + flag_cols],
        on=["firm_id", "year"],
        how="left"
    )
    # True wenn PO-Daten unvollständig: buybacks fehlen, aber Dividenden vorhanden
    out["po_incomplete"] = out["buybacks_missing"] & ~out["dividends_missing"]
    n_incomplete = out["po_incomplete"].sum()
    print(f"po_incomplete (buybacks fehlen, Dividenden vorhanden): {n_incomplete:,} ({n_incomplete/len(out):.1%})")
else:
    print("WARN: buybacks_missing/dividends_missing nicht im master-Panel verfügbar")

# ----------------------------
# 7.1c tv_share_suspect: TV_share > 1
# ----------------------------
# THEORETISCHE BEGRÜNDUNG: TV_share > 1 ist theoretisch unmöglich.
#
# Per Konstruktion:
#   Total PV = Σ_{h=1}^{H} PV(PO_h)  +  PV(Terminal Value)
#   TV_share = PV(TV) / Total PV  →  muss im Intervall [0, 1] liegen.
#
# TV_share > 1 ⟺ Σ_{h=1}^{H} PV(PO_h) < 0 (negative explizite PV-Summe).
#
# MÖGLICHE URSACHEN (empirisch zu unterscheiden via dr − beg_ss):
#
# (A) Fast-Divergenz: dr ≈ beg_ss → Terminwert TV = PO_{H+1}/(dr−beg_ss−v2)
#     explodiert numerisch. Erkennbar an: dr − beg_ss sehr klein (< 1–2bp).
#
# (B) Negative explizite Payouts: Bei Wachstumsfirmen mit hohem erwartetem
#     beg (Eigenkapitalwachstum >> Ausschüttungen) ist E[PO_h] < 0 für
#     viele h = 1..H (Firma investiert mehr als sie ausschüttet, csprof < beg).
#     Die kumulierte explizite PV-Summe wird negativ, obwohl dr weit von
#     beg_ss entfernt ist und die geometrische Reihe konvergiert.
#     Dies ist das numerisch häufigere Muster, insbesondere bei H=30.
#
# In beiden Fällen werden diese Beobachtungen via duration_usable ausgeschlossen.
# ----------------------------
out["tv_share_suspect"] = out["TV_share"] > 1.0
n_suspect = int(out["tv_share_suspect"].sum())
print(f"tv_share_suspect (TV_share > 1): {n_suspect:,} / {len(out):,} obs "
      f"({n_suspect/len(out):.1%})")

# THESIS: Diagnostik — Ursache von TV_share > 1
# Unterscheidet Ursache A (Fast-Divergenz, dr ≈ beg_ss)
# von Ursache B (negative explizite PV-Summe, dr weit von beg_ss)
if n_suspect > 0 and "discount_rate_NP" in out.columns:
    _tv_sus = out.loc[out["tv_share_suspect"] & out["discount_rate_NP"].notna()].copy()
    if len(_tv_sus) > 0:
        _tv_sus["dr_minus_beg_ss"] = _tv_sus["discount_rate_NP"] - beg_ss
        print(f"\nTHESIS: Verteilung (discount_rate_NP − beg_ss) bei TV_share > 1 "
              f"(n = {len(_tv_sus):,}):")
        print(_tv_sus["dr_minus_beg_ss"].describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95]).round(4))
        _near_div     = (_tv_sus["dr_minus_beg_ss"].abs() < 0.01).mean()
        _median_spread = _tv_sus["dr_minus_beg_ss"].median()
        print(f"\nAnteil |dr − beg_ss| < 1bp (Ursache A, Fast-Divergenz): {_near_div:.1%}")
        if _near_div < 0.10 and _median_spread > 0.02:
            print(f"→ Fast-Divergenz ist NICHT die Hauptursache.")
            print(f"  Median dr − beg_ss = {_median_spread:.3f} — deutlich > 0, "
                  f"geometrische Reihe konvergiert.")
            print(f"→ Wahrscheinlichere Ursache (B): negative kumulierte explizite PV-Summe.")
            print(f"  Wachstumsfirmen mit E[csprof_h] < E[beg_h] für h = 1..{H}:")
            print(f"  erwartete Netto-Ausschüttungen sind negativ → Σ PV(PO_h) < 0.")
        else:
            print(f"→ Fast-Divergenz (Ursache A) nicht auszuschließen: "
                  f"{_near_div:.1%} der Fälle nahe Divergenzgrenze.")
        print(f"→ Alle {n_suspect:,} tv_share_suspect-Fälle werden via "
              f"duration_usable korrekt ausgeschlossen.")
        del _tv_sus

# ----------------------------
# 7.1d Kombinierter duration_usable-Flag
# ----------------------------
out["duration_usable"] = (
    ~out["dr_constrained"]
    & ~out["pv_fit_poor"]
    & ~out["tv_share_suspect"]
    & out["Duration_NetPayout"].notna()
)
n_usable = int(out["duration_usable"].sum())
print(f"duration_usable (alle Flags): {n_usable:,} ({n_usable/len(out):.1%})")

# ----------------------------
# 7.2 Winsorise Duration at 99th percentile
# Extreme tail values (up to 2000+ years) are numerical artefacts
# from firms where cumBEg barely exceeds dr; cap them before saving.
# ----------------------------
dur_p99 = out["Duration_NetPayout"].quantile(0.99)
n_clipped = (out["Duration_NetPayout"] > dur_p99).sum()
out["Duration_NetPayout"] = out["Duration_NetPayout"].clip(upper=dur_p99)
print(f"Duration_NetPayout winsorised at P99 = {dur_p99:.1f} years "
      f"({n_clipped} obs clipped)")

# ----------------------------
# 7.3 Coverage summary and save
# ----------------------------
n_total = len(out)
n_dur   = out["Duration_NetPayout"].notna().sum()
n_clean = out["duration_usable"].sum()
print(f"\nOutput rows : {n_total:,}  ({out['firm_id'].nunique():,} firms × {out['year'].nunique()} years)")
print(f"Duration_NetPayout coverage (all)    : {n_dur:,} / {n_total:,} ({n_dur/n_total:.1%})")
print(f"Duration_NetPayout coverage (usable) : {n_clean:,} / {n_total:,} ({n_clean/n_total:.1%})")
print(f"Year range  : {int(out['year'].min())} – {int(out['year'].max())}")
print()
print(out[["firm_id", "year", "Duration_NetPayout", "discount_rate_NP", "TV_share"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(3))

save_parquet(out, "EQDuration_Netpayout")
print("\nSaved: EQDuration_Netpayout.parquet")
print("Neue Spalten: buybacks_missing, dividends_missing, po_incomplete,")
print("              Duration_NP_divonly, pv_fit_poor, tv_share_suspect, duration_usable")

# TV_share: Normal vs. Suspect
fig, ax = plt.subplots(figsize=(8, 3))
out.loc[~out["tv_share_suspect"], "TV_share"].clip(upper=1.5).hist(
    bins=50, ax=ax, alpha=0.7, label="Normal (TV_share ≤ 1)", density=True
)
out.loc[out["tv_share_suspect"], "TV_share"].clip(upper=5).hist(
    bins=30, ax=ax, alpha=0.7, label="Suspect (TV_share > 1)", density=True
)
ax.set_xlabel("TV_share")
ax.set_title("TV_share: Normal vs. Suspect Beobachtungen")
ax.legend()
plt.tight_layout()
plt.show()

print(
    "\nEmpfehlung Regressionen:\n"
    "  df_clean = out[out['duration_usable']]\n"
    "  (entspricht: ~dr_constrained & ~pv_fit_poor & ~tv_share_suspect & Duration notna)"
)


## 9. Diagnostics of Duration_NP

Single consolidated diagnostics block (distribution, correlations, cross-sectional checks, robust visualization, and floor-vs-non-floor comparison).


In [ ]:

with pd.option_context("display.max_rows", 200, "display.max_columns", None, "display.width", 200):


    print("\n==============================")
    print("1. Basic Duration Distribution")
    print("==============================")

    print(state_panel["Duration_NP"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))
    print("\nNaN share:", state_panel["Duration_NP"].isna().mean())

    d_tmp = pd.to_numeric(state_panel["Duration_NP"], errors="coerce")
    d_tmp = d_tmp[np.isfinite(d_tmp)]
    if len(d_tmp) > 0:
        print(f"Share Duration_NP > 100: {(d_tmp > 100).mean():.2%}")
        print(f"Share Duration_NP > 200: {(d_tmp > 200).mean():.2%}")

    if "discount_rate_NP" in state_panel.columns:
        dr = pd.to_numeric(state_panel["discount_rate_NP"], errors="coerce")
        dr_valid = dr.dropna()
        if len(dr_valid) > 0:
            near_gss = (dr_valid <= (beg_ss + 1e-3)).mean()
            near_low = (dr_valid <= 0.01).mean()
            print(f"Share discount_rate_NP <= beg_ss+1bp: {near_gss:.2%}")
            print(f"Share discount_rate_NP <= 1%: {near_low:.2%}")


    # ------------------------------------------------------------------
    # 1b. THESIS: Duration-Niveau — Vergleich mit Gonçalves (2020)
    # ------------------------------------------------------------------
    # Gonçalves (2020) berichtet für eine US-Stichprobe (CRSP, 1963–2014):
    #   Median ≈ 39 Jahre, Dezil 1 ≈ 17 Jahre, Dezil 10 ≈ 105 Jahre
    # Dieses Notebook zeigt für Euro Stoxx 500:
    #   Median ≈ 23 Jahre, P95 ≈ 32 Jahre
    #
    # Strukturelle Gründe für niedrigere europäische Duration:
    #   (a) Niedrigere Ausschüttungsrenditen in Europa → mechanisch kürzere Duration,
    #       da ein größerer Anteil des Firmenwerts in der fernen Zukunft liegt, wenn
    #       Ausschüttungen gering sind (Boudoukh et al. 2007, Table 1).
    #   (b) Euro Stoxx 500 ist bankenlastig (Finanzsektor ~20%); Banken haben
    #       strukturell kurze implizite Duration durch regulatorische Ausschüttungs-
    #       beschränkungen und zyklische Dividendenpolitik.
    #   (c) Kürzere Stichprobenhistorie beeinflusst VAR-Steady-State-Kalibrierung
    #       (weniger Beobachtungen für expanding-window VAR in frühen Jahren).
    #   (d) Gormsen (2020, "Dividend Growth over the Business Cycle") dokumentiert
    #       international niedrigere realisierte Dividendenwachstumsraten außerhalb
    #       der USA, was zu niedrigerer impliziter Duration führt.
    #
    # FAZIT: Dies ist KEIN Fehler, sondern eine bekannte strukturelle Differenz.
    # ------------------------------------------------------------------
    _d_clean = d_tmp[np.isfinite(d_tmp)] if len(d_tmp) > 0 else pd.Series(dtype=float)
    _med  = float(_d_clean.median()) if len(_d_clean) > 0 else float("nan")
    _mean = float(_d_clean.mean())   if len(_d_clean) > 0 else float("nan")
    _p5   = float(_d_clean.quantile(0.05)) if len(_d_clean) > 0 else float("nan")
    _p95  = float(_d_clean.quantile(0.95)) if len(_d_clean) > 0 else float("nan")

    print("\n============================================================")
    print("THESIS: Duration-Niveau — Vergleich Gonçalves (2020) vs. diese Stichprobe")
    print("============================================================")
    _comp = pd.DataFrame({
        "Stichprobe":  ["Gonçalves (2020) USA", "Dieses Notebook (Euro Stoxx 500)"],
        "Mittelwert":  ["~42 Jahre",            f"{_mean:.1f} Jahre"],
        "Median":      ["~39 Jahre",            f"{_med:.1f} Jahre"],
        "P5":          ["~14 Jahre",            f"{_p5:.1f} Jahre"],
        "P95":         ["~80 Jahre",            f"{_p95:.1f} Jahre"],
    }).set_index("Stichprobe")
    print(_comp.to_string())
    print()
    print("Strukturelle Gründe für niedrigere europäische Duration:")
    print("  (a) Niedrigere Ausschüttungsrenditen in Europa → mechanisch kürzere Duration")
    print("  (b) Bankenlastiger Sektor-Mix im Euro Stoxx 500 (kurze implizite Duration)")
    print("  (c) Kürzere Stichprobenhistorie beeinflusst VAR-Steady-State")
    print("  (d) Gormsen (2020): niedrigere realisierte Dividendenwachstumsraten außerhalb USA")
    print("→ Kein Methodenfehler; bekannte strukturelle Differenz US vs. Europa.")
    print("============================================================")


    print("\n==============================")
    print("2. Correlations with Key Variables")
    print("==============================")

    vars_check = [
        "Duration_NP",
        "bm",
        "py",
        "ag",
        "beg",
        "roe",
        "csprof"
    ]

    corr = state_panel[vars_check].corr().round(3)
    display(corr)


    print("\n==============================")
    print("3. Duration by BM Quintiles")
    print("==============================")

    bm_test = state_panel.groupby(
        pd.qcut(state_panel["bm"], 5), observed=False
    )["Duration_NP"].mean()

    display(bm_test.to_frame("Duration_NP").round(4))


    print("\n==============================")
    print("4. Duration by Asset Growth Quintiles")
    print("==============================")

    growth_test = state_panel.groupby(
        pd.qcut(state_panel["ag"], 5), observed=False
    )["Duration_NP"].mean()

    display(growth_test.to_frame("Duration_NP").round(4))


    print("\n==============================")
    print("5. Duration by Payout Yield Quintiles")
    print("==============================")

    payout_test = state_panel.groupby(
        pd.qcut(state_panel["py"], 5), observed=False
    )["Duration_NP"].mean()

    display(payout_test.to_frame("Duration_NP").round(4))


    print("\n==============================")
    print("6. Duration Histogram")
    print("==============================")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        state_panel["Duration_NP"].dropna(),
        bins=60,
        color=COLORS["blue_light"],
        edgecolor=COLORS["primary"],
        alpha=0.9,
    )
    ax.set_title("Distribution of Equity Duration")
    ax.set_xlabel("Duration (years)")
    ax.set_ylabel("Frequency")
    style_axes(ax)
    plt.tight_layout()
    plt.show()


    print("\n==============================")
    print("7. Duration vs BM: Robust Visual Diagnostic")
    print("==============================")

    plot_df = state_panel[["bm", "Duration_NP", "TV_share", "discount_rate_NP"]].dropna(subset=["bm", "Duration_NP"]).copy()

    if len(plot_df) == 0:
        print("No valid observations for bm vs Duration_NP plot.")
    else:
        # Report concentration in the visually suspicious low-duration band.
        band_share = plot_df["Duration_NP"].between(16, 20).mean()
        print(f"Share with Duration_NP in [16, 20]: {band_share:.2%}")

        # Trim only for visualization (core diagnostics still use full sample).
        bm_lo, bm_hi = plot_df["bm"].quantile([0.01, 0.99])
        dur_lo, dur_hi = plot_df["Duration_NP"].quantile([0.01, 0.99])
        plot_trim = plot_df[
            plot_df["bm"].between(bm_lo, bm_hi)
            & plot_df["Duration_NP"].between(dur_lo, dur_hi)
        ].copy()

        sample_n = min(6000, len(plot_trim))
        plot_sample = plot_trim.sample(n=sample_n, random_state=42) if sample_n > 0 else plot_trim

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        hb = axes[0].hexbin(
            plot_trim["bm"],
            plot_trim["Duration_NP"],
            gridsize=45,
            mincnt=1,
            cmap="Blues"
        )
        axes[0].set_title("BM vs Duration (1-99% trimmed, hexbin)")
        axes[0].set_xlabel("Book-to-Market")
        axes[0].set_ylabel("Duration")
        style_axes(axes[0], grid_axis="both")
        cbar_left = fig.colorbar(hb, ax=axes[0], label="Count")
        cbar_left.outline.set_edgecolor("#C4CDD7")

        if "TV_share" in plot_sample.columns and plot_sample["TV_share"].notna().any():
            sc = axes[1].scatter(
                plot_sample["bm"],
                plot_sample["Duration_NP"],
                c=plot_sample["TV_share"],
                s=12,
                alpha=0.35,
                cmap="cividis"
            )
            cbar_right = fig.colorbar(sc, ax=axes[1], label="TV_share")
            cbar_right.outline.set_edgecolor("#C4CDD7")
            axes[1].set_title("Sample scatter colored by TV_share")
        else:
            axes[1].scatter(
                plot_sample["bm"],
                plot_sample["Duration_NP"],
                s=12,
                alpha=0.35,
                color=COLORS["accent"],
            )
            axes[1].set_title("Sample scatter")

        axes[1].set_xlabel("Book-to-Market")
        axes[1].set_ylabel("Duration")
        style_axes(axes[1], grid_axis="both")

        plt.tight_layout()
        plt.show()


In [ ]:

# ============================================================
# STEP 8: Diagnostik – TV_share Verteilung
# Zeigt nur die korrekt berechnete TV_share aus Step 6.
# compute_tv_share_diag() wurde entfernt: sie verwendete eine
# andere Methode (kein Jensen, kein e_po, r=0.08 hartkodiert)
# und der Vergleich war methodisch nicht aussagekräftig.
# ============================================================

print("\n==============================")
print("Terminal Value Share (aus Step 6, firm-year-spezifisches dr)")
print("==============================")
print(state_panel["TV_share"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))
print(f"\nAnteil TV_share > 1 (suspect): "
      f"{(state_panel['TV_share'] > 1.0).mean():.2%}")

fig, ax = plt.subplots(figsize=(7, 4))
state_panel["TV_share"].clip(upper=1.0).dropna().hist(
    bins=50, ax=ax,
    color=COLORS["orange"],
    edgecolor=COLORS["primary"],
    alpha=0.85
)
ax.set_title("Terminal Value Share (auf [0,1] gekappt für Visualisierung)")
ax.set_xlabel("PV(Terminal Value) / Total PV")
ax.set_ylabel("Frequency")
style_axes(ax)
plt.tight_layout()
plt.show()


### Step 9: Validierung — Kumulativer Payout nach Duration-Dezil


In [ ]:
# ============================================================
# STEP 9: Validierung — Kumulativer Payout-Anteil nach Duration-Dezil
# Repliziert Paper Abbildung 1(a): "% of ME paid within h years"
# ============================================================

val_panel = state_panel[
    state_panel["Duration_NP"].notna() & state_panel["ME"].notna()
].copy()

val_panel["dur_decile"] = pd.qcut(
    val_panel["Duration_NP"], 10,
    labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    duplicates="drop"
)
val_panel["dur_decile"] = pd.to_numeric(val_panel["dur_decile"], errors="coerce")

H_val = 10

def compute_cumulative_po_fraction(panel, H=H_val):
    """
    Für jedes Firmenjahr (firm_id, year=t): kumulativer PO-Anteil
    der nächsten H Jahre relativ zu ME_t.
    """
    p = panel[["firm_id", "year", "PO", "ME", "dur_decile"]].copy()
    p = p.sort_values(["firm_id", "year"])
    records = []
    for h in range(1, H + 1):
        tmp = p.copy()
        tmp["PO_fwd"] = tmp.groupby("firm_id")["PO"].shift(-h)
        tmp = tmp.dropna(subset=["PO_fwd", "ME", "dur_decile"])
        tmp["po_frac_h"] = tmp["PO_fwd"] / tmp["ME"]
        tmp["horizon"] = h
        records.append(tmp[["firm_id", "year", "horizon", "po_frac_h", "dur_decile"]])
    return pd.concat(records, ignore_index=True)

fwd_df = compute_cumulative_po_fraction(val_panel, H=H_val)
fwd_df = fwd_df.sort_values(["firm_id", "year", "horizon"])
fwd_df["po_frac_cum"] = fwd_df.groupby(["firm_id", "year"])["po_frac_h"].cumsum()

plot_val = (
    fwd_df.groupby(["dur_decile", "horizon"])["po_frac_cum"]
    .mean()
    .reset_index()
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
highlight_deciles = [1, 3, 5, 8, 10]
colors_val = plt.cm.Blues(np.linspace(0.3, 0.9, len(highlight_deciles)))

for i, dec in enumerate(highlight_deciles):
    sub = plot_val[plot_val["dur_decile"] == dec]
    ax.plot(
        sub["horizon"],
        sub["po_frac_cum"] * 100,
        marker="o", markersize=4,
        color=colors_val[i],
        label=f"Decile {dec}"
    )

ax.set_xlabel("Years after Duration Measurement (h)")
ax.set_ylabel("Cumulative PO / ME at t  (%)")
ax.set_title(
    "Validation: Cumulative Net Payout as % of Market Equity\n"
    "by Duration Decile  (replicates Gonçalves 2020, Fig. 1a)"
)
ax.legend(title="Duration Decile", loc="upper left")
ax.set_xticks(range(1, H_val + 1))
style_axes(ax)
plt.tight_layout()
plt.show()

# Validierungscheck
d1_h10  = plot_val.query("dur_decile==1  and horizon==@H_val")["po_frac_cum"].values
d10_h10 = plot_val.query("dur_decile==10 and horizon==@H_val")["po_frac_cum"].values
if len(d1_h10) and len(d10_h10):
    print(f"\nValidierung (Horizont h={H_val}):")
    print(f"  Dezil 1 (kurz):  {d1_h10[0]*100:.1f}% des ME ausgeschüttet")
    print(f"  Dezil 10 (lang): {d10_h10[0]*100:.1f}% des ME ausgeschüttet")
    if d1_h10[0] > d10_h10[0]:
        print("  Erwartetes Muster bestätigt: kurze Duration → höherer kurzfristiger Payout")
    else:
        print("  WARNUNG: Unerwartetes Muster — Duration sortiert nicht in der erwarteten Richtung")


### Thesis-Tabelle: Datenverfügbarkeit und Robustheit


In [ ]:
# ============================================================
# THESIS-AUSGABE: Datenverfügbarkeit und Robustheit
# ============================================================

_div_dur = state_panel[["Duration_NP", "Duration_NP_divonly"]].corr().iloc[0, 1] \
           if "Duration_NP_divonly" in state_panel.columns else float("nan")

_po_incomp_col = "po_incomplete" if "po_incomplete" in out.columns else None
_bb_miss_col   = "buybacks_missing" if "buybacks_missing" in master.columns else None

print("=" * 55)
print("DATENVERFÜGBARKEIT — THESIS-TABELLE")
print("=" * 55)
print(f"Firmenjahre gesamt:                    {len(master):>8,}")
if _bb_miss_col:
    print(f"davon Buyback-Daten vorhanden:         {(~master[_bb_miss_col]).sum():>8,}  ({(~master[_bb_miss_col]).mean():.1%})")
if _po_incomp_col:
    _n_inco = out[_po_incomp_col].sum()
    print(f"davon nur Dividenden (kein Buyback):   {_n_inco:>8,}  ({_n_inco/len(out):.1%})")
_n_dur  = out["Duration_NetPayout"].notna().sum()
print(f"Duration_NP berechnet (gesamt):        {_n_dur:>8,}  ({_n_dur/len(out):.1%})")
if "dr_constrained" in out.columns:
    _n_clean = (~out["dr_constrained"] & out["Duration_NetPayout"].notna()).sum()
    print(f"Duration_NP berechnet (clean):         {_n_clean:>8,}  ({_n_clean/len(out):.1%})")
print()
print("Robustheit: Korr(Duration_NP, Duration_NP_divonly)")
print(f"  → {_div_dur:.3f}" if not pd.isna(_div_dur) else "  → n/a (Robustheitslauf nicht ausgeführt)")



## 10. Temporal Diagnostics

Diese Sektion analysiert die zeitliche Entwicklung der Duration-Messung und entspricht
Gonçalves (2020), Tabelle 6 und Abbildung IA.3:

- **10a:** Coverage von `duration_usable` nach Jahr (Balkendiagramm + Firmenzahl)
- **10b:** Querschnittliche Streuung σ(log Duration) — Prädiktor für die Short-Duration-Prämie
- **10c:** Duration-Dezil-Mediane über die Zeit (Dezile 1, 5, 10)


In [ ]:

# ============================================================
# STEP 10a: Coverage über die Zeit
# Anteil duration_usable-Beobachtungen und Firmenzahl pro Jahr
# ============================================================

if "duration_usable" in out.columns:
    _cov_yr = (
        out.groupby("year")
        .agg(
            usable_share=("duration_usable", "mean"),
            firm_count=("firm_id", "nunique"),
        )
        .reset_index()
    )

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()

    ax1.bar(
        _cov_yr["year"], _cov_yr["usable_share"] * 100,
        color=COLORS.get("blue_light", "#6baed6"), alpha=0.75,
        label="Anteil usable (%)"
    )
    ax2.plot(
        _cov_yr["year"], _cov_yr["firm_count"],
        color=COLORS.get("primary", "#1f4e79"), linewidth=2,
        marker="o", markersize=4, label="Firmenzahl"
    )

    ax1.set_xlabel("Jahr")
    ax1.set_ylabel("Anteil duration_usable (%)")
    ax2.set_ylabel("Firmenzahl im Panel")
    ax1.set_ylim(0, 105)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
    ax1.set_title("Share of Usable Duration Observations by Year")
    style_axes(ax1)
    plt.tight_layout()
    plt.show()

    print("THESIS: Coverage duration_usable nach Jahr:")
    print(_cov_yr.set_index("year").round(3).to_string())
    del _cov_yr
else:
    print("SKIP: duration_usable nicht in out verfügbar. Erst Step 7 ausführen.")



### 10b. Querschnittliche Streuung σ(log Duration) über die Zeit

Analog zu Gonçalves (2020), Tabelle 6 und Abbildung IA.3.
σ(log Duration) misst, wie weit Duration-Werte im Querschnitt spreizen — ein Prädiktor für die Höhe der Short-Duration-Prämie in der späteren Prämienregression.

**Hinweis:** Euro-spezifische Rezessionsperioden (z.B. Eurokrise 2011–12, COVID-19 2020)
sind nicht schattiert, da kein NBER-äquivalentes Indikator für Europa verwendet wird.


In [ ]:

# ============================================================
# STEP 10b: Querschnittliche Streuung σ(log Duration) über die Zeit
# Gonçalves (2020), Tabelle 6 und Abbildung IA.3
# Höhere σ(log Duration) → größere erwartete Short-Duration-Prämie
# ============================================================

_dur_col  = "Duration_NetPayout" if "Duration_NetPayout" in out.columns else "Duration_NP"
_flag_col = "duration_usable"    if "duration_usable"    in out.columns else None

_sigma_src = out.loc[
    (out[_flag_col] if _flag_col else out[_dur_col].notna()) & out[_dur_col].notna()
].copy()
_sigma_src["log_dur"] = np.log(_sigma_src[_dur_col].clip(lower=0.1))

_sigma_yr = _sigma_src.groupby("year")["log_dur"].std().reset_index()
_sigma_yr.columns = ["year", "sigma_log_dur"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    _sigma_yr["year"], _sigma_yr["sigma_log_dur"],
    color=COLORS.get("primary", "#1f4e79"), linewidth=2, marker="o", markersize=4
)
ax.set_xlabel("Jahr")
ax.set_ylabel("σ(log Duration_NP)")
ax.set_title("Cross-Sectional Dispersion σ(log Duration) by Year")
ax.annotate(
    "Rezessionsperioden nicht markiert\n(keine NBER-Daten für Euro-Stichprobe verfügbar)",
    xy=(0.02, 0.96), xycoords="axes fraction",
    fontsize=8, color="gray", va="top"
)
style_axes(ax)
plt.tight_layout()
plt.show()

print("THESIS: σ(log Duration) nach Jahr:")
print(_sigma_yr.set_index("year").round(4).to_string())
del _sigma_src, _sigma_yr


### 10c. Duration-Dezil-Mediane über die Zeit

Median-Duration für Dezil 1 (kurze Duration), Dezil 5 (Mitte) und Dezil 10 (lange Duration) pro Jahr.
Zeigt, ob der Duration-Spread stabil oder zeitvariierend ist — eine Voraussetzung für die Prämienregression.


In [ ]:

# ============================================================
# STEP 10c: Duration-Dezil-Mediane über die Zeit
# ============================================================

_dur_col  = "Duration_NetPayout" if "Duration_NetPayout" in out.columns else "Duration_NP"
_flag_col = "duration_usable"    if "duration_usable"    in out.columns else None

_decile_src = out.loc[
    (out[_flag_col] if _flag_col else out[_dur_col].notna()) & out[_dur_col].notna()
].copy()

# Jährliche Dezilzuweisung (Querschnitt)
_decile_src["dur_decile"] = _decile_src.groupby("year")[_dur_col].transform(
    lambda x: pd.qcut(x, 10, labels=range(1, 11), duplicates="drop")
)
_decile_src["dur_decile"] = pd.to_numeric(_decile_src["dur_decile"], errors="coerce")

_decile_med = (
    _decile_src[_decile_src["dur_decile"].isin([1, 5, 10])]
    .groupby(["year", "dur_decile"])[_dur_col]
    .median()
    .reset_index()
    .pivot(index="year", columns="dur_decile", values=_dur_col)
)
_decile_med.columns = [f"Dezil {int(c)}" for c in _decile_med.columns]

_colors_dec = [
    COLORS.get("blue_light", "#6baed6"),
    COLORS.get("primary",    "#1f4e79"),
    COLORS.get("orange",     "#e07b39"),
]
fig, ax = plt.subplots(figsize=(10, 4))
for _col, _clr in zip(_decile_med.columns, _colors_dec):
    ax.plot(_decile_med.index, _decile_med[_col],
            linewidth=2, marker="o", markersize=4, color=_clr, label=_col)
ax.set_xlabel("Jahr")
ax.set_ylabel("Median Duration (Jahre)")
ax.set_title("Duration-Dezil-Mediane über die Zeit\n(jährliche Querschnittsdezile)")
ax.legend(title="Dezil")
style_axes(ax)
plt.tight_layout()
plt.show()

print("THESIS: Duration-Dezil-Mediane nach Jahr:")
print(_decile_med.round(1).to_string())
del _decile_src, _decile_med


 ## 11. Duration Sensitivity Analysis

**Part A — DR-Floor Sensitivität**: Wie sensitiv ist `Duration_NP` gegenüber der unteren Schranke des Bisektions-Solvers? Baseline vs. 4%, 6%, 8% Mindest-Diskontrate.

**Part B — Expected Payback Period (EPP)**: Diskontierungsfreies Alternativmaß — kleinster Horizont *h*, bei dem kumulierte undiskontierte Jensen-korrigierte erwartete Ausschüttungen die Marktkapitalisierung erreichen (Gonçalves 2020, Eq. 14). Kein DR-Solving erforderlich.

 ### Part B: Expected Payback Period (EPP)

EPP (Expected Payback Period) is computed in EQDuration_Robustness.ipynb
and stored in EQDuration_Robustness.parquet. It was moved there to keep
all robustness duration measures in a single output file.